# 5G Phased Array Coverage Simulator — Real-Time Beam Steering

Fully self-contained — one floorplan upload, mask auto-generated internally. Run cells in order (top to bottom).

**Pipeline:** single-antenna baseline → NodePlacerOptimizer finds weak-spot nodes → **frame 0: node placements shown on their own** → baseline coverage heatmap → phased array installed at the same position → **all 73 beam angles (0°-355°, 5° steps) precomputed once** → loop: find weakest remaining node, step through cached frames 5° at a time toward it (rounding to the nearest cached angle), mark it served, repeat → **final frame: full 360° max-hold coverage map**.

**Please type frequency WITH units** (e.g. `50MHz`, `2.4GHz`) — a bare number will prompt you to confirm the unit rather than guessing. Default frequency is 50MHz, matching the phased-array solver's own default -- pick a frequency your mask's pixel resolution can actually resolve (the notebook will warn you if it can't).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 1 — IMPORTS + SHARED UTILITIES                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import sys, subprocess, importlib, importlib.util

_REQUIRED = ["numpy", "scipy", "matplotlib", "PIL", "cv2", "skimage"]
_missing = [p for p in _REQUIRED if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q",
         "numpy", "scipy", "matplotlib", "Pillow", "opencv-python",
         "scikit-image"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import os
import io
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Wedge
from scipy.sparse import spdiags, eye, kron
from scipy.sparse.linalg import factorized
from scipy import ndimage
import cv2
from PIL import Image

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("✅  Imports ready.")

# ═══════════════════════════════════════════════════════════════════════════
#  COORDINATE CONVENTION — ONE grid, ONE convention, used EVERYWHERE.
# ═══════════════════════════════════════════════════════════════════════════
#
#  Every array in this notebook (the building mask, the single-antenna RSSI
#  grid, and the phased-array field grid) shares the EXACT SAME shape:
#  (rows, cols) = binary_mask.shape. Origin TOP-LEFT, y increases DOWNWARD.
#    meters -> pixel:   col = round(x_m / h),   row = round(y_m / hy)
#    pixel -> meters:   x_m = col * h,           y_m = row * hy
#  where h = real_width/cols, hy = real_height/rows.
# ═══════════════════════════════════════════════════════════════════════════

SPONGE_THICKNESS = 20   # single source of truth. Both solvers + node-placer
                         # exclusion logic all read this constant instead of
                         # each hardcoding their own copy of "20".

ETA0    = 120 * np.pi   # free-space impedance, ≈377 Ω — shared by BOTH
C_LIGHT = 3e8            # solvers so their RSSI physics is identical.


def meters_to_pixel(x_m, y_m, h, hy):
    """meters (top-left origin, y-down) -> (row, col) pixel index."""
    col = int(round(x_m / h))
    row = int(round(y_m / hy))
    return row, col


def pixel_to_meters(row, col, h, hy):
    """(row, col) pixel index -> meters (top-left origin, y-down)."""
    return col * h, row * hy


def shortest_angular_path(theta_from, theta_to):
    """Shortest direction/distance from theta_from to theta_to on a circle.
    Returns (direction, distance) where direction: +1 = clockwise
    (increasing angle), -1 = counter-clockwise (decreasing angle).
    Test: 350 -> 10 should return (+1, 20), i.e. +20 degrees, not -340.
    """
    diff = (theta_to - theta_from) % 360
    if diff > 180:
        return -1, 360 - diff
    return 1, diff


# ── Self-test for shortest_angular_path ─────────────────────────────────────
_dir, _dist = shortest_angular_path(350, 10)
assert (_dir, _dist) == (1, 20), f"shortest_angular_path broken: got {(_dir, _dist)}"
print("✅  shortest_angular_path self-test passed (350°→10° == +20°).")
def compute_grid_size(img_shape, cell_budget=260000):
    """
    v22: adopted from wifi_sim_engine/stage1_engine.py's
    compute_grid_size() (same logic/math) -- allocates rows/cols matching
    the source image's aspect ratio while keeping the total cell count
    close to cell_budget. This is the piece that was MISSING before: the
    solver was running at the uploaded floorplan's full raw pixel count
    (a 1067x1474 upload -> 1.57M cells), which is both needlessly slow
    AND, combined with a 0.5 m/px cell size, blew the real-world map size
    out to 500-700m for a campus-block-sized image -- explains why the
    single-antenna heatmap looked almost entirely dead (signal correctly
    decaying to nothing over a map that was accidentally far too large).
    Default cell_budget=260000 matches the reference exactly.
    """
    img_h, img_w = img_shape[0], img_shape[1]
    aspect = img_h / img_w
    cols = int(round(np.sqrt(cell_budget / aspect)))
    rows = int(round(aspect * cols))
    rows = max(rows, 10)
    cols = max(cols, 10)
    return rows, cols


def downsample_mask_to_grid(hard_mask, rows, cols):
    """
    v22: adopted from wifi_sim_engine/stage1_engine.py's
    set_mask_from_canonical() (same logic/math) -- takes an already-clean
    canonical boolean mask (ours comes from SmartMapPreprocessor, not
    discarded, just downsampled AFTER its full-resolution cleanup) and
    anti-aliased-resizes it down to the solver's (rows, cols) grid.
    Returns (occupancy, hard_mask_downsampled):
      - occupancy: smooth float [0,1] per cell -- true fractional building
        occupancy from proper anti-aliased downsampling, used for
        effective-medium k-field blending in build_helmholtz_solver.
        (Replaces the earlier Gaussian-blur-of-a-hard-mask approximation
        from last round -- this is the reference's actual method, not an
        approximation of it.)
      - hard_mask_downsampled: occupancy > 0.5, used for source-snapping,
        node placement, and frame backdrops at the solver's resolution.
    """
    occupancy = cv2.resize(hard_mask.astype(np.float32), (cols, rows),
                            interpolation=cv2.INTER_AREA)
    hard_mask_downsampled = occupancy > 0.5
    return occupancy, hard_mask_downsampled


print("✅  Grid-sizing utilities loaded (compute_grid_size, "
      "downsample_mask_to_grid -- same logic/math as reference).")


# ═══════════════════════════════════════════════════════════════════════════
#  SHARED HELMHOLTZ SYSTEM MATRIX — used by BOTH SingleAntennaSim and
#  PhasedArraySim. Previously each class assembled its own copy of this
#  exact same matrix (identical Laplacian, identical k-field, identical
#  sponge layer, since both solvers share the same mask/frequency/
#  absorption) and ran its OWN sparse LU factorization — i.e. the single
#  heaviest step in the whole notebook was being done twice for no reason.
#  Now built ONCE in main() and the same factorized solve function is
#  handed to both solvers. Math is byte-for-byte identical to before —
#  only the duplication is removed.
# ═══════════════════════════════════════════════════════════════════════════

def build_helmholtz_solver(occupancy, real_width, f_sim,
                            alpha_air, alpha_eff, n_eff=2.0,
                            sponge_thickness=SPONGE_THICKNESS,
                            pml_max_loss=0.5):  # v24 FIX: reference default is 0.5, not 0.3
    """
    Assemble the sparse Helmholtz matrix and return its LU factorization
    plus the grid params both solvers need. Depends ONLY on occupancy/
    real_width/f_sim/alpha_air/alpha_eff/n_eff -- NOT on source position
    or count -- so one instance is valid for both the single-antenna
    solve and every phased-array steering angle.

    v18: adopts the reference engine's (wifi_sim_engine/stage1_engine.py)
    field-solve approach, same logic/math, on top of our own grid:
      - EFFECTIVE-MEDIUM BOUNDARY BLENDING: instead of a hard binary step
        (k0 | k_building) at every building edge, blend per-cell using a
        fractional occupancy in [0,1]: k_cell = (1-f)*k0 + f*k_building.
      - QUADRATIC PML RAMP: absorbing boundary loss ramps smoothly from 0
        at the interior edge of the sponge band up to pml_max_loss at the
        outer edge, matching the reference's ramp exactly.

    v22: `occupancy` is now the REAL thing, not an approximation of it.
    Earlier this took a hard 0/1 mask and faked soft edges with a Gaussian
    blur; now it takes the true smooth [0,1] occupancy that
    downsample_mask_to_grid() produces via proper anti-aliased resizing
    (same method as the reference's set_mask_from_canonical) -- no blur
    step needed here anymore, occupancy arrives pre-blended.
    """
    rows, cols = occupancy.shape
    N = rows * cols
    h = real_width / cols   # meters/pixel (square-pixel assumption,
                             # matches original single-h convention)
    k0 = 2 * np.pi * f_sim / 3e8

    e    = np.ones(N)
    D2_r = spdiags([e, -2*e, e], [-1, 0, 1], rows, rows) / (h**2)
    D2_c = spdiags([e, -2*e, e], [-1, 0, 1], cols, cols) / (h**2)
    Laplacian = kron(eye(cols), D2_r) + kron(D2_c, eye(rows))

    k_air      = k0 - 1j * alpha_air
    k_building = k0 * n_eff - 1j * alpha_eff

    occ_flat = occupancy.flatten(order="F")
    k_vec = (1.0 - occ_flat) * k_air + occ_flat * k_building

    # Quadratic-ramped PML, matching the reference engine exactly:
    # ramp = clip((sponge_thickness - dist_to_edge)/sponge_thickness, 0, 1)**2
    row_idx = np.arange(rows).reshape(-1, 1)
    col_idx = np.arange(cols).reshape(1, -1)
    dist_to_edge = np.minimum.reduce([
        np.broadcast_to(row_idx, (rows, cols)),
        np.broadcast_to((rows - 1) - row_idx, (rows, cols)),
        np.broadcast_to(col_idx, (rows, cols)),
        np.broadcast_to((cols - 1) - col_idx, (rows, cols)),
    ]).astype(float)
    ramp = np.clip((sponge_thickness - dist_to_edge) / sponge_thickness, 0.0, 1.0) ** 2
    extra_loss = (pml_max_loss * ramp).flatten(order="F")
    k_vec = k_vec - 1j * extra_loss

    A = Laplacian + spdiags(k_vec**2, 0, N, N)

    print("   > Factorizing shared Helmholtz system matrix "
          "(ONE factorization, reused by single-antenna AND phased-array solvers)...")
    solve = factorized(A.tocsc())

    shared_params = {"rows": rows, "cols": cols, "h": h, "k0": k0}
    return solve, shared_params

print("✅  Shared Helmholtz solver builder loaded.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 2 — SmartMapPreprocessor (backend layer 1: color-based mask      ║
# ║   generation from an uploaded floorplan image)                          ║
# ║                                                                          ║
# ║   Integrated per the earlier plan: this REPLACES the project's old      ║
# ║   flat-darkness-threshold mask generator (removed from the input-       ║
# ║   collection cell below, not kept side-by-side, so there is no          ║
# ║   overlapping/duplicate classifier logic in the notebook). Validated    ║
# ║   round-trip against three real test maps: catches park/green space,   ║
# ║   strips Arabic+English text labels, and preserves real courtyards      ║
# ║   instead of solid-filling them.                                        ║
# ║                                                                          ║
# ║   Output convention already matches the rest of this project exactly:  ║
# ║   BinaryMask == 1 -> building (signal blocker), 0 -> air/street/park.  ║
# ║   No conversion needed anywhere downstream.                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
"""
SmartMapPreprocessor - Python/Colab port of SmartMapPreprocessor.m

Mask convention (matches MATLAB source, NOT the user's verbal description):
    BinaryMask == 1 -> Buildings (signal blockers)
    BinaryMask == 0 -> Streets / Air / Background / Parks / Water

Display only: showResults() plots ~BinaryMask, so on screen buildings appear
BLACK and streets/air appear WHITE, exactly like the MATLAB showResults().

CHANGES vs the original version (color classification only -- label/icon/pin
removal, hole-fill, and display are untouched):

  1. Background threshold widened from a strict ">240 on all 3 channels" to
     ">=230" AND "low color saturation" (channels close to each other). The
     old cutoff was tight enough that a real pale-white building fill
     (common in Google-Maps-style exports) could tip into "background" and
     get eaten, leaving holes/gaps in that building's mask.
  2. NEW explicit park/green-space bucket. Previously anything that wasn't
     white background or blue-tinted street fell into "building" by
     default -- confirmed on a real test map that this misclassifies park
     areas (grass, green space) as solid buildings/signal-blockers, which
     is wrong. Green space is now excluded the same way streets are.
     Rule: G is clearly the dominant channel (greener than both red and
     blue) -- catches Google Maps' pastel-green park fill without an exact
     hex match, since Google's exact greens drift slightly across
     zoom/render/region and an exact-hex rule would be fragile.
  3. Everything left after excluding background + street/water + green is
     treated as building, same as before -- this is intentionally a
     positive-by-elimination rule (not a positive building-color match),
     since this project's target is simply "building vs not-building" and
     campus building fills vary in shade (yellow, tan, near-white) too much
     for one fixed building-color match to catch reliably.

CHANGES (round 2/3) -- text/label removal simplified, then split into two
stages for Arabic (thin) vs English (bolder) strokes:
  4. _remove_labels() replaced entirely. The old version ran a per-blob
     regionprops loop checking solidity/extent/euler-number/pin-holes --
     complex, and still left jagged text spikes fused onto buildings.
     New approach: text/icon strokes are consistently THIN vs. solid
     building wall fill. A morphological opening wipes anything thinner
     than a threshold radius; a size filter mops up leftover fragments.
  5. Single opening radius wasn't enough: Arabic text strokes and English
     letters render at different thicknesses on this map, so one radius
     left English labels behind (or shaved buildings if pushed wider).
     Now TWO passes: a narrow opening (3px) over the whole mask first
     (clears thin/Arabic strokes), then a wider opening (5px) applied
     ONLY to whatever is still small afterward -- not the whole mask
     again -- so bolder English leftovers get cleared without a second,
     more destructive pass over confirmed-real (large) building blocks.
     Tested on the real map: stage 2 makes ~1,800 additional pixel
     changes beyond stage 1 alone, concentrated in label-sized regions,
     not inside the big building blobs.
     This is intentionally NOT exact, same trade-off as before.
"""

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.measure import label, regionprops


class SmartMapPreprocessor:
    def __init__(self, image_path):
        self.ImagePath = image_path
        self.OriginalImage = np.array(Image.open(image_path).convert('RGB'))
        self.BinaryMask = None
        self.RealWidth_m = None
        self.RealHeight_m = None

    def process(self, width_m=None, height_m=None):
        print("   > Processing Map Image...")

        img = self.OriginalImage
        if img.ndim == 3 and img.shape[2] >= 3:
            R = img[:, :, 0].astype(float)
            G = img[:, :, 1].astype(float)
            B = img[:, :, 2].astype(float)

            # Step A: Background (pure white / very bright gray). Kept at the
            # ORIGINAL >240 cutoff -- tried widening this to catch pale-white
            # buildings, but tested it against the real map and it backfired:
            # this map's building WALLS render at ~232-238 (light steel
            # gray), almost the same as background gray, so widening the
            # cutoff ate ~67% of real building area. Left as-is; the
            # pale-white-building risk is a smaller, unconfirmed edge case,
            # not worth breaking a confirmed-working rule for.
            is_bg = (R > 240) & (G > 240) & (B > 240)

            # Step B: Streets / water (blue tint: B noticeably higher than R)
            is_street = (B > R + 10) & (B > 150)

            # Step C: Parks / green space -- NEW, confirmed bug fix. Tested
            # on the real map: previously the park area (top-right) fell
            # into "building" by default since it's neither white bg nor
            # blue street. Rule below caught it cleanly (verified: the
            # park region is now 100% excluded, 0 leftover pixels).
            # Not an exact hex match, since Google's exact park green drifts
            # slightly across zoom/render/region.
            is_green = (G > R + 12) & (G > B + 12) & (G > 150)

            # Step D: Buildings = everything else -> 1 (True)
            self.BinaryMask = ~(is_bg | is_street | is_green)
        else:
            gray = img.astype(float)
            if gray.ndim == 3:
                gray = gray.mean(axis=2)
            self.BinaryMask = gray < 220

        # Morphological cleanup
        # Remove small noise blobs (text, icons, stray pixels), like bwareaopen(mask, 50)
        self.BinaryMask = remove_small_objects(self.BinaryMask, min_size=50)

        # Remove text labels: small blobs, low fill-ratio (glyph strokes), isolated
        self.BinaryMask = self._remove_labels(self.BinaryMask, self.OriginalImage)

        # Fill holes inside buildings, like imfill(mask, 'holes')
        # Fill holes, but NOT indiscriminately -- a real courtyard (a
        # building with an open interior, confirmed on a real map: ~847px
        # hole) must NOT be filled in, only small gaps from broken/dashed
        # outlines or anti-aliasing noise (everything else measured on that
        # same map topped out at 156px) should be. See _fill_small_holes().
        self.BinaryMask = self._fill_small_holes(self.BinaryMask)

        # Real-world dimensions: Colab has no inputdlg -> pass as args or prompt via input()
        if width_m is None or height_m is None:
            try:
                w = input("Enter Map Width in meters [500]: ").strip()
                h = input("Enter Map Height in meters [500]: ").strip()
                self.RealWidth_m = float(w) if w else 500.0
                self.RealHeight_m = float(h) if h else 500.0
            except Exception:
                print("Dimension input canceled. Using default 500x500 meters.")
                self.RealWidth_m = 500.0
                self.RealHeight_m = 500.0
        else:
            self.RealWidth_m = float(width_m)
            self.RealHeight_m = float(height_m)

        print(f"   > Map set to {self.RealWidth_m:.1f} m x {self.RealHeight_m:.1f} m.")

    def _fill_small_holes(self, mask, max_hole_area=300):
        """
        Fill enclosed background regions inside a building, but ONLY if
        they're small (broken-outline gaps, anti-aliasing noise). Larger
        enclosed regions are left alone -- they're real courtyards, not
        gaps, and blanket-filling them (the old behavior) turned a hollow
        building into a solid block.

        max_hole_area=300 was picked by measuring actual hole sizes on a
        real map with a confirmed courtyard: the courtyard was 847px, and
        every other (non-courtyard, gap-noise) hole on that same map was
        <=156px -- 300 sits cleanly between the two with margin either way.
        """
        filled = ndimage.binary_fill_holes(mask)
        holes = filled & ~mask
        lbl = label(holes, connectivity=2)
        out = mask.copy()
        for rp in regionprops(lbl):
            if rp.area <= max_hole_area:
                out[lbl == rp.label] = True
        return out

    def _remove_labels(self, mask, original_image,
                        open_radius_narrow=3, open_radius_wide=5,
                        small_blob_area=400, min_object_size=40):
        """
        Strip text-label / thin-icon strokes from a building mask
        (True=building). Simple, efficient, NOT exact -- traded on purpose
        for speed/maintainability per project requirement ("acceptable,
        not 100%").

        TWO-STAGE cascade (per your observation: Arabic strokes are
        thinner, English letters render bolder/wider on this map style,
        so one opening radius doesn't clear both):

          Stage 1 -- narrow opening (open_radius_narrow, default 3px)
          applied to the WHOLE mask. Clears thin strokes (mostly Arabic
          text, thin icon lines) while leaving solid building blocks
          intact.

          Stage 2 -- wider opening (open_radius_wide, default 5px) applied
          ONLY to whatever is still small after stage 1
          (< small_blob_area px, i.e. NOT a confirmed real building).
          This clears bolder leftover strokes (mostly English text) without
          re-running the wider, more destructive filter over big real
          building blocks, which would shave real thin walls.

        Trade-offs (accepted): a genuinely thin building wing under
        open_radius_narrow can get shaved in stage 1; a small real
        building under small_blob_area could get over-eroded in stage 2.
        `original_image` kept for interface compatibility (unused).
        """
        # Stage 1: narrow opening, whole mask -- clears thin (Arabic) strokes
        stage1 = binary_opening(mask, footprint=disk(open_radius_narrow))
        stage1 = remove_small_objects(stage1, min_size=min_object_size)

        # Stage 2: wide opening, but ONLY on parts still small after stage 1
        # -- clears bolder (English) leftover strokes without touching
        # confirmed-real (large) building blocks a second time.
        lbl = label(stage1, connectivity=2)
        small_mask = np.zeros_like(stage1, dtype=bool)
        big_mask = np.zeros_like(stage1, dtype=bool)
        for p in regionprops(lbl):
            if p.area < small_blob_area:
                small_mask[lbl == p.label] = True
            else:
                big_mask[lbl == p.label] = True

        small_cleaned = binary_opening(small_mask, footprint=disk(open_radius_wide))
        small_cleaned = remove_small_objects(small_cleaned, min_size=min_object_size)

        return big_mask | small_cleaned

    def showResults(self):
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        fig.patch.set_facecolor('w')
        fig.canvas.manager.set_window_title('Smart Map Preprocessor Results') \
            if fig.canvas.manager else None

        axes[0].imshow(self.OriginalImage)
        axes[0].set_title('Original Map')
        axes[0].axis('off')

        # Inverted for display only (buildings black, streets/air white),
        # matching MATLAB's imshow(~obj.BinaryMask)
        axes[1].imshow(~self.BinaryMask, cmap='gray')
        axes[1].set_title('Generated Binary Mask')
        axes[1].set_xlabel('Black = Buildings (0) | White = Streets/Air (1)')
        axes[1].set_xticks([])
        axes[1].set_yticks([])

        plt.tight_layout()
        plt.show()




print("✅  SmartMapPreprocessor loaded (validated: park/green exclusion, "
      "two-stage Arabic/English text removal, courtyard-preserving hole-fill).")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 2 — PhasedArraySim CLASS (REAL-TIME STEERING)                    ║
# ║                                                                          ║
# ║   FIX (this round): PhasedArraySim used to build its source amplitude   ║
# ║   from an uncalibrated empirical "MATLAB formula" constant and return   ║
# ║   raw 20*log10(|E|) — RELATIVE dB, explicitly not comparable to         ║
# ║   SingleAntennaSim's calibrated TRUE dBm. That was the cause of the     ║
# ║   phased-array heatmap looking unrealistically stronger.                ║
# ║                                                                          ║
# ║   Now PhasedArraySim uses the EXACT SAME method as SingleAntennaSim:    ║
# ║     - source_amp per element = sqrt(2*pi*eta0*Pt*Gt)      (real physics)║
# ║     - calibrated against the same closed-form free-space reference     ║
# ║       (same calib_ratio SingleAntennaSim computes for this grid --      ║
# ║       valid to reuse because it depends only on grid/frequency/         ║
# ║       absorption, not on source amplitude or position -- the field is  ║
# ║       linear in source_amp so the ratio theory/measured cancels out)    ║
# ║     - solve_for_angle() now runs the array's superposed field through  ║
# ║       the SAME RSSI chain (2D->3D correction, power density, effective ║
# ║       aperture, Friis) SingleAntennaSim uses, returning TRUE dBm.       ║
# ║   Array physics (phase_shifter, element spacing, steering) UNCHANGED.  ║
# ╚══════════════════════════════════════════════════════════════════════════╝


class PhasedArraySim:
    """
    5-element phased array Helmholtz simulator (FDM, 2D). Returns TRUE dBm,
    calibrated the same way as SingleAntennaSim, so the two are directly
    comparable.

    Parameters
    ----------
    raw_image    : PIL.Image   — Color map image (for display overlay)
    binary_mask  : np.ndarray  — Shape (rows, cols). 1=building, 0=air
    real_width   : float       — Physical width  of the map in meters
    real_height  : float       — Physical height of the map in meters
    tx_power_dbm : float       — PER-ELEMENT transmit power in dBm.
    tx_gain_dbi  : float       — PER-ELEMENT transmit gain, dBi (default 2.15,
                                 dipole -- same default SingleAntennaSim uses,
                                 so the two are apples-to-apples).
    rx_gain_dbi  : float       — receive antenna gain, dBi (default 0.0)
    antenna_x    : float|None  — Array start X in meters from left edge
    antenna_y    : float|None  — Array start Y in meters from top edge
    f_sim        : float       — Simulation frequency in Hz
    alpha_air    : float       — Air absorption coefficient
    alpha_eff    : float       — Building absorption coefficient
    step_angle   : int         — Beam step in degrees (default 5)
    shared_solve, shared_grid_params : pre-built factorized solver from
                   build_helmholtz_solver() -- same one SingleAntennaSim uses.
    calib_ratio  : float       — the SAME free-space calibration ratio
                   SingleAntennaSim computed for this grid/frequency/
                   absorption (see SingleAntennaSim._calibrate_source_amp).
                   Required so both solvers' dBm numbers are on one scale.
    """

    def __init__(self, raw_image, binary_mask, real_width, real_height,
                 tx_power_dbm=22,  # v24: matches reference sourceValue default exactly
                 tx_gain_dbi=2.15,
                 rx_gain_dbi=0.0,
                 antenna_x=None,
                 antenna_y=None,
                 f_sim=50e6,
                 alpha_air=0.00002,
                 alpha_eff=0.062,
                 step_angle=5,
                 shared_solve=None,
                 shared_grid_params=None,
                 calib_ratio=1.0):

        self.img_raw    = np.array(raw_image)
        self.map_mask   = binary_mask
        self.params     = self._get_hyperparameters(
            binary_mask, real_width, real_height,
            tx_power_dbm, tx_gain_dbi, rx_gain_dbi, antenna_x, antenna_y,
            f_sim, alpha_air, alpha_eff, step_angle, calib_ratio
        )
        self.img_cropped = None
        self.src_x       = None
        self.src_y       = None
        self._solve      = shared_solve
        self._shared_grid_params = shared_grid_params
        self._r_grid_m   = None   # per-pixel range from array ref point, cached
        self.sectors_array = []
        self.angle_cache   = {}
        self.result_matrix = None

    # ─────────────────────────────────────────────────────────────────────
    def _get_hyperparameters(self, binary_mask, real_width, real_height,
                              tx_power_dbm, tx_gain_dbi, rx_gain_dbi,
                              antenna_x, antenna_y,
                              f_sim, alpha_air, alpha_eff, step_angle,
                              calib_ratio):
        p = {}

        p["rows"], p["cols"] = binary_mask.shape
        p["real_width"]  = real_width
        p["real_height"] = real_height
        p["h"]           = real_width / p["cols"]
        p["hy"]          = real_height / p["rows"]

        p["f_sim"]   = f_sim
        p["c"]       = 3e8
        p["lambda"]  = p["c"] / p["f_sim"]
        p["k0"]      = 2 * np.pi * p["f_sim"] / p["c"]

        p["alpha_air"] = alpha_air
        p["n_eff"]     = 2.0
        p["alpha_eff"] = alpha_eff

        p["num_elements"] = 5
        p["d"]            = p["lambda"] / 2

        p["step_angle"] = step_angle

        if antenna_x is None:
            antenna_x = real_width  * 0.05
        if antenna_y is None:
            antenna_y = real_height * 0.05

        p["antenna_x_m"] = antenna_x
        p["antenna_y_m"] = antenna_y

        # ── Same physics-based source amplitude + calibration method as ────
        # SingleAntennaSim, PER ELEMENT (each of the 5 elements individually
        # radiates tx_power_dbm -- unchanged intent from before).
        p["user_tx_power_dbm"] = tx_power_dbm
        p["tx_gain_dbi"]       = tx_gain_dbi
        p["rx_gain_dbi"]       = rx_gain_dbi
        p["array_gain_db"]     = 20 * np.log10(p["num_elements"])   # informational only

        # v24 FIX: same change as SingleAntennaSim -- no Pt/Gt formula, no
        # free-space calib_ratio rescale. Raw amplitude per element,
        # exactly like the reference. calib_ratio arg kept for call-site
        # compatibility but no longer applied.
        Gr = 10 ** (rx_gain_dbi / 10)
        p["Gr"] = Gr
        p["source_amp"] = float(tx_power_dbm)
        p["calib_ratio"] = calib_ratio

        p["keep_width_ratio"] = 1.0
        return p

    # ─────────────────────────────────────────────────────────────────────
    def process_map(self):
        """Crop map to active columns (keep_width_ratio = 1.0 by default)."""
        valid_cols        = round(self.params["cols"] * self.params["keep_width_ratio"])
        self.img_cropped  = self.img_raw[:, :valid_cols, :]
        self.map_mask     = self.map_mask[:, :valid_cols]
        self.params["cols"] = valid_cols
        return self

    # ─────────────────────────────────────────────────────────────────────
    def place_sources(self):
        """Place 5 antenna elements starting at the user-defined position."""
        h_x = self.params["real_width"]  / self.params["cols"]
        h_y = self.params["real_height"] / self.params["rows"]

        start_x = round(self.params["antenna_x_m"] / h_x)
        start_y = round(self.params["antenna_y_m"] / h_y)

        pixel_spacing = max(1, round(self.params["d"] / h_x))

        if round(self.params["d"] / h_x) < 1:
            print(f"   ⚠️  WARNING: Element spacing d = {self.params['d']:.3f} m is smaller "
                  f"than one pixel ({h_x:.3f} m). Elements will overlap. "
                  f"Consider a higher-resolution map or lower frequency.")

        self.src_x = np.zeros(self.params["num_elements"], dtype=int)
        self.src_y = np.zeros(self.params["num_elements"], dtype=int)
        for n in range(self.params["num_elements"]):
            self.src_x[n] = start_x + n * pixel_spacing
            self.src_y[n] = start_y

        return self

    # ─────────────────────────────────────────────────────────────────────
    def phase_shifter(self, theta_deg):
        """
        θ=0°   → broadside (beam perpendicular to array axis)
        θ=90°  → endfire right
        θ=270° → endfire left
        """
        theta_rad    = np.deg2rad(theta_deg)
        n            = np.arange(self.params["num_elements"])
        phase_shifts = -n * self.params["k0"] * self.params["d"] * np.sin(theta_rad)
        return self.params["source_amp"] * np.exp(1j * phase_shifts)

    # ─────────────────────────────────────────────────────────────────────
    def _build_system_matrix(self):
        """Reuse the shared, pre-built factorized solve if given (same one
        SingleAntennaSim uses); otherwise build locally as a fallback."""
        if self._solve is not None:
            return self._solve

        self._solve, self._shared_grid_params = build_helmholtz_solver(
            self.map_mask, self.params["real_width"], self.params["f_sim"],
            self.params["alpha_air"], self.params["alpha_eff"],
            self.params["n_eff"])
        return self._solve

    # ─────────────────────────────────────────────────────────────────────
    def _range_grid_m(self):
        """Per-pixel range (meters) from the array's reference point
        (antenna_x_m, antenna_y_m -- same convention SingleAntennaSim uses
        for its own single source), cached after first computation."""
        if self._r_grid_m is not None:
            return self._r_grid_m

        p = self.params
        rows, cols = p["rows"], p["cols"]
        h, hy = p["h"], p["hy"]

        col_idx = np.arange(cols) * h
        row_idx = np.arange(rows) * hy
        X_m, Y_m = np.meshgrid(col_idx, row_idx)

        r = np.sqrt((X_m - p["antenna_x_m"])**2 + (Y_m - p["antenna_y_m"])**2)
        r_min = max(min(h, hy), 1e-6)
        self._r_grid_m = (np.maximum(r, r_min), r < r_min, r_min)
        return self._r_grid_m

    # ─────────────────────────────────────────────────────────────────────
    def solve_for_angle(self, theta_deg: float) -> np.ndarray:
        """
        Compute the field for ONE steering angle and convert it to the
        SAME field-magnitude-dB scale as SingleAntennaSim.rssi_dbm_grid
        (20*log10(|E|), Gaussian-smoothed) -- so this is directly
        comparable to the single-antenna baseline, not a separate scale.
        """
        if self._solve is None:
            self._build_system_matrix()

        p = self.params
        rows, cols = p["rows"], p["cols"]
        N    = rows * cols
        k    = p["k0"]
        lam  = p["lambda"]
        Gr   = p["Gr"]

        src_indices = self.src_y + self.src_x * rows

        b = np.zeros(N, dtype=complex)
        b[src_indices] = self.phase_shifter(theta_deg)

        E_vec = self._solve(-b)
        E     = E_vec.reshape((rows, cols), order="F")

        # v23 FIX: same change as SingleAntennaSim.rssi_dbm_grid -- plain
        # field-magnitude dB (matches wifi_sim_engine/stage1_engine.py),
        # not a second Friis/effective-aperture link-budget stacked on
        # top of the FDM solve. Gaussian-smoothed (sigma=1.5), same as
        # the reference and same as the single-antenna baseline, so both
        # solvers are on the identical scale and directly comparable.
        E_mag = np.abs(E)
        mag_db = 20 * np.log10(E_mag + 1e-12)
        mag_db = ndimage.gaussian_filter(mag_db, sigma=1.5)
        return mag_db

    # ─────────────────────────────────────────────────────────────────────
    def precompute_all_angles(self):
        """Sweep ALL angles 0°->355° in step_angle increments ONCE, cache
        every heatmap (now true dBm). Factorized matrix built/reused once."""
        if self._solve is None:
            self._build_system_matrix()

        step = self.params["step_angle"]
        angles = list(range(0, 360, step))
        self.sectors_array = []
        self.angle_cache = {}

        print(f"   > Precomputing all {len(angles)} beam angles "
              f"(0°-355°, step={step}°), true-dBm RSSI chain...")
        for theta in angles:
            mag_db = self.solve_for_angle(theta)
            self.sectors_array.append({"angle": theta, "matrix_db": mag_db})
            self.angle_cache[theta] = mag_db
        print(f"   > Done. {len(angles)} angle frames cached.")
        return self

    # ─────────────────────────────────────────────────────────────────────
    def get_cached_angle(self, theta_deg):
        """Look up a precomputed heatmap for the nearest cached angle."""
        step = self.params["step_angle"]
        rounded = int(round(theta_deg / step) * step) % 360
        return rounded, self.angle_cache[rounded]

    # ─────────────────────────────────────────────────────────────────────
    def max_hold_coverage(self):
        """Best true-dBm value at each pixel across all precomputed angles."""
        all_db = np.stack([s["matrix_db"] for s in self.sectors_array], axis=2)
        self.result_matrix = np.max(all_db, axis=2)
        return self.result_matrix

    # ─────────────────────────────────────────────────────────────────────
    def run_simulation(self, angles_to_show=None):
        self.process_map()
        self.place_sources()
        self._build_system_matrix()
        self.precompute_all_angles()

        if angles_to_show:
            print("\n--- Plotting sample cached angles for a quick sanity check ---")
            n = len(angles_to_show)
            fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
            if n == 1:
                axes = [axes]
            for ax, theta in zip(axes, angles_to_show):
                _, mag_db = self.get_cached_angle(theta)
                vmax = max(15, np.max(mag_db))
                im = ax.imshow(mag_db, cmap="jet", vmin=-80, vmax=vmax)
                ax.set_title(f"Angle: {theta}°", fontsize=10)
                ax.axis("off")
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            plt.suptitle("Sample Beam Angle Coverage Maps", fontsize=13)
            plt.tight_layout()
            plt.show()

        return self


print("✅  PhasedArraySim (true-dBm, calibrated same as SingleAntennaSim) loaded.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 3 — SINGLE-ANTENNA SOLVER  (baseline, true dBm)                  ║
# ║                                                                          ║
# ║   Physics UNCHANGED: f0 = sqrt(2*pi*eta0*Pt*Gt), 2D->3D correction,     ║
# ║   power density, effective aperture, RSSI conversion, calibration       ║
# ║   against the closed-form free-space field — all identical to before.   ║
# ║                                                                          ║
# ║   CHANGED (bug fix only): _build_system_matrix no longer assembles its  ║
# ║   own duplicate Laplacian/sponge/k-field — accepts an optional shared,  ║
# ║   pre-built solve (see build_helmholtz_solver in Cell 1), same one used ║
# ║   by PhasedArraySim, so this matrix is factorized ONCE and shared.      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ETA0 and C_LIGHT are defined once in Cell 1 and shared by both solvers.


class SingleAntennaSim:
    """
    Single omnidirectional-antenna Helmholtz coverage solver.
    Grid, coordinate convention, and FDM assembly are IDENTICAL to
    PhasedArraySim.

    Parameters
    ----------
    binary_mask   : np.ndarray  — Shape (rows, cols). 1=building, 0=air.
    real_width    : float — meters
    real_height   : float — meters
    tx_power_dbm  : float — transmit power, dBm
    tx_gain_dbi   : float — transmit antenna gain, dBi (default 2.15, dipole)
    rx_gain_dbi   : float — receive antenna gain, dBi (default 0.0, isotropic)
    antenna_x     : float — meters from LEFT edge
    antenna_y     : float — meters from TOP edge
    f_sim         : float — frequency, Hz
    alpha_air     : float — air absorption coefficient
    alpha_eff     : float — building absorption coefficient
    shared_solve, shared_grid_params : optional pre-built factorized solver
                   from build_helmholtz_solver() — pass these to avoid a
                   redundant factorization when reused by PhasedArraySim.
    """

    def __init__(self, binary_mask, real_width, real_height,
                 tx_power_dbm=22.0,  # v24: matches reference stage1_engine.py sourceValue default exactly
                 tx_gain_dbi=2.15,
                 rx_gain_dbi=0.0,
                 antenna_x=None,
                 antenna_y=None,
                 f_sim=50e6,
                 alpha_air=0.00002,
                 alpha_eff=0.062,
                 shared_solve=None,
                 shared_grid_params=None):

        self.map_mask = binary_mask
        self.params = self._get_hyperparameters(
            binary_mask, real_width, real_height, tx_power_dbm, tx_gain_dbi,
            rx_gain_dbi, antenna_x, antenna_y, f_sim, alpha_air, alpha_eff)
        self._solve = shared_solve
        self._shared_grid_params = shared_grid_params
        self.src_row = None
        self.src_col = None

    # ─────────────────────────────────────────────────────────────────────
    def _get_hyperparameters(self, binary_mask, real_width, real_height,
                              tx_power_dbm, tx_gain_dbi, rx_gain_dbi,
                              antenna_x, antenna_y, f_sim, alpha_air, alpha_eff):
        p = {}

        p["rows"], p["cols"] = binary_mask.shape
        p["real_width"]  = real_width
        p["real_height"] = real_height
        p["h"]  = real_width / p["cols"]
        p["hy"] = real_height / p["rows"]

        p["f_sim"]  = f_sim
        p["c"]      = C_LIGHT
        p["lambda"] = p["c"] / p["f_sim"]
        p["k0"]     = 2 * np.pi * p["f_sim"] / p["c"]

        ppw_effective = p["lambda"] / min(p["h"], p["hy"])
        p["ppw_effective"] = ppw_effective
        if ppw_effective < 10:
            print(f"   ⚠️  WARNING: effective grid resolution is only "
                  f"{ppw_effective:.1f} points/wavelength (mask pixel size "
                  f"{min(p['h'], p['hy']):.3f} m vs λ={p['lambda']:.3f} m). "
                  f"Values <10 may show numerical dispersion artifacts.")

        p["alpha_air"] = alpha_air
        p["n_eff"]     = 2.0
        p["alpha_eff"] = alpha_eff

        if antenna_x is None:
            antenna_x = real_width * 0.5
        if antenna_y is None:
            antenna_y = real_height * 0.5
        p["antenna_x_m"] = antenna_x
        p["antenna_y_m"] = antenna_y

        p["tx_power_dbm"] = tx_power_dbm
        p["tx_gain_dbi"]  = tx_gain_dbi
        p["rx_gain_dbi"]  = rx_gain_dbi
        # v24 FIX: this arg is kept named tx_power_dbm for UI/backward-compat,
        # but the reference engine (stage1_engine.py build_params) never
        # converts a dBm value into a physical source amplitude at all --
        # it feeds a RAW numeric amplitude ("sourceValue", default 22)
        # straight into the FDM source term, with no Pt/Gt formula and no
        # free-space calibration step. That Pt/Gt/calibration chain here
        # was the reason peak kept coming out far below the reference's
        # (e.g. 2.3 dB instead of 8.4 dB at "tx_power_dbm"=10) -- it was
        # rescaling the amplitude DOWN to match a closed-form free-space
        # target the reference never targets in the first place.
        # Now: source_amp = tx_power_dbm, used directly as the raw FDM
        # amplitude, exactly like the reference's sourceValue. Set this to
        # 22 (the default) to reproduce the reference's own numbers
        # exactly; raise/lower it to scale peak strength up/down linearly.
        Gr = 10 ** (rx_gain_dbi / 10)
        p["Gr"] = Gr
        p["source_amp"] = float(tx_power_dbm)

        return p

    # ─────────────────────────────────────────────────────────────────────
    def place_source(self):
        """Antenna position (meters, top-left/y-down) -> (row, col) index.
        v18: adopts the reference engine's nearest-open-cell snapping --
        if the requested position lands inside a building, search
        outward ring-by-ring for the closest air cell instead of solving
        with a source buried in a wall (same logic as
        wifi_sim_engine/stage1_engine.py's _nearest_open_cell)."""
        p = self.params
        row, col = meters_to_pixel(p["antenna_x_m"], p["antenna_y_m"],
                                    p["h"], p["hy"])
        row = int(np.clip(row, 0, p["rows"] - 1))
        col = int(np.clip(col, 0, p["cols"] - 1))

        self.was_source_snapped = False
        if self.map_mask[row, col]:
            row, col = self._nearest_open_cell(row, col)
            self.was_source_snapped = True
            print(f"   ⚠️  Requested antenna position is inside a building -- "
                  f"snapped to the nearest open (air) cell.")

        self.src_row, self.src_col = row, col
        # v24: reference engine does NOT calibrate source_amp against a
        # closed-form free-space target -- it uses the raw sourceValue
        # amplitude as-is. Calibration call removed to match exactly.
        return self

    # ─────────────────────────────────────────────────────────────────────
    def _nearest_open_cell(self, row0, col0):
        """Ring-search outward from (row0, col0) for the nearest cell where
        map_mask is False (air). Same logic as the reference engine."""
        rows, cols = self.map_mask.shape
        max_radius = max(rows, cols)
        for radius in range(1, max_radius):
            r_lo, r_hi = max(0, row0 - radius), min(rows - 1, row0 + radius)
            c_lo, c_hi = max(0, col0 - radius), min(cols - 1, col0 + radius)
            ring_r, ring_c = np.meshgrid(np.arange(r_lo, r_hi + 1),
                                          np.arange(c_lo, c_hi + 1), indexing="ij")
            on_ring = ((ring_r == r_lo) | (ring_r == r_hi) |
                       (ring_c == c_lo) | (ring_c == c_hi))
            cand_r, cand_c = ring_r[on_ring], ring_c[on_ring]
            open_mask = ~self.map_mask[cand_r, cand_c]
            if np.any(open_mask):
                open_r, open_c = cand_r[open_mask], cand_c[open_mask]
                dist = (open_r - row0) ** 2 + (open_c - col0) ** 2
                best = np.argmin(dist)
                return int(open_r[best]), int(open_c[best])
        return row0, col0

    # ─────────────────────────────────────────────────────────────────────
    def _calibrate_source_amp(self):
        """
        Rescale source_amp so this FDM solver's field magnitude matches the
        closed-form free-space |E_2D(r)| = f0/sqrt(8*pi*k*r) a few pixels
        from the source (angularly averaged to avoid an interference null).
        """
        p = self.params
        rows, cols = p["rows"], p["cols"]
        N = rows * cols
        h, hy = p["h"], p["hy"]
        k = p["k0"]

        src_index = self.src_row + self.src_col * rows

        b = np.zeros(N, dtype=complex)
        b[src_index] = p["source_amp"]

        solve = self._build_system_matrix()
        E = solve(-b).reshape((rows, cols), order="F")

        Y, X = np.meshgrid(np.arange(rows), np.arange(cols), indexing="ij")
        R_m = np.sqrt(((X - self.src_col) * h) ** 2 +
                       ((Y - self.src_row) * hy) ** 2)

        px = min(h, hy)
        r_tests = px * np.array([6, 8, 10, 12])
        meas, theory = [], []
        for r_test in r_tests:
            ring = np.abs(R_m - r_test) < px
            if not np.any(ring):
                continue
            meas.append(np.abs(E[ring]).mean())
            theory.append(p["source_amp"] / np.sqrt(8 * np.pi * k * r_test))

        if meas:
            calib_ratio = float(np.mean(np.array(theory) / np.array(meas)))
        else:
            calib_ratio = 1.0

        p["source_amp"] *= calib_ratio
        p["_calibration_ratio"] = calib_ratio

    # ─────────────────────────────────────────────────────────────────────
    def _build_system_matrix(self):
        """
        Return the factorized Helmholtz system matrix. If a shared,
        pre-built solve was passed in at construction time, reuse it — no
        rebuild, no second LU factorization. Otherwise build locally
        (fallback for standalone use outside the shared-solver pipeline).
        """
        if self._solve is not None:
            return self._solve

        self._solve, self._shared_grid_params = build_helmholtz_solver(
            self.map_mask, self.params["real_width"], self.params["f_sim"],
            self.params["alpha_air"], self.params["alpha_eff"],
            self.params["n_eff"])
        return self._solve

    # ─────────────────────────────────────────────────────────────────────
    def solve(self) -> np.ndarray:
        """Run the solve and return the raw complex 2D field E_2D."""
        if self._solve is None:
            self._build_system_matrix()

        p = self.params
        rows, cols = p["rows"], p["cols"]
        N = rows * cols
        src_index = self.src_row + self.src_col * rows

        b = np.zeros(N, dtype=complex)
        b[src_index] = p["source_amp"]

        E_vec = self._solve(-b)
        return E_vec.reshape((rows, cols), order="F")

    # ─────────────────────────────────────────────────────────────────────
    def rssi_dbm_grid(self) -> np.ndarray:
        """
        v23 FIX: back to the reference engine's actual method
        (wifi_sim_engine/stage1_engine.py render_heatmap_png) -- plain
        field-magnitude dB, mag_db = 20*log10(|E| + eps), then Gaussian-
        smoothed (sigma=1.5) exactly like the reference. This is NOT a
        second physical Friis/effective-aperture link-budget on top of
        the already-physical FDM solve -- the v21/v22 "true dBm" chain
        (2D->3D r^-1/2 correction, power density, effective aperture,
        Friis) double-counted geometric spreading loss that the FDM
        solve already models, which is why the baseline heatmap was
        collapsing to near-total dead zone (median ~-80 dBm, min below
        -200 dBm) instead of matching the reference's healthy-looking
        coverage (median ~-62 dB, peak ~+8 to +9 dB -- verified against
        the reference's own screenshot: "True Peak: 8.4 dB").
        tx_power_dbm/tx_gain_dbi still drive source_amp (via
        _calibrate_source_amp), so transmit power still meaningfully
        changes the result -- only the DOUBLE post-hoc RSSI conversion
        is removed, matching the reference exactly.
        """
        E2D = self.solve()
        mag_db = 20 * np.log10(np.abs(E2D) + 1e-12)
        # Gaussian smoothing (display/metric smoothing only, does not
        # change the underlying field solve) -- same sigma=1.5 the
        # reference applies before rendering, erases FDM grid-staircase
        # and destructive-interference-null speckle.
        mag_db = ndimage.gaussian_filter(mag_db, sigma=1.5)
        return mag_db

    # ─────────────────────────────────────────────────────────────────────
    def run(self):
        self.place_source()
        return self.rssi_dbm_grid()


print("✅  SingleAntennaSim (shared-solver aware) loaded.")


## NodePlacerOptimizer (v18: faithful port from wifi_sim_engine/node_placer_optimizer.py -- sector clustering + polar transform + real ILP set-cover, same logic/math as the reference)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 5 — NodePlacerOptimizer                                          ║
# ║                                                                          ║
# ║   v18: replaced entirely with the faithful port from                    ║
# ║   wifi_sim_engine/node_placer_optimizer.py, per explicit request to     ║
# ║   take "exactly the same logic and math" for node placement +           ║
# ║   optimization. Interface was already compatible with this project's    ║
# ║   run_node_placement() call site (Cell 7): run(campusImg, stage1_out,   ║
# ║   binaryMask) with stage1_out={"mag_db","src_x","src_y"}, returning     ║
# ║   finalNodes as (x,y) pairs -- zero changes needed at the call site.    ║
# ║   Math/logic below is UNCHANGED from the reference: sector-based        ║
# ║   clustering + polar transform, real ILP set-cover (scipy.optimize.     ║
# ║   milp) for unconstrained placement, budget-constrained greedy ILP      ║
# ║   variant, RF-sector-std pruning, spatial-diversity enforcement,        ║
# ║   strict-air-and-boundary filtering. Far more sophisticated than the    ║
# ║   previous simplified greedy-prune version this replaces.               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
"""
Faithful Python port of attached_assets/NodePlacerOptimizer_1784764881666.m.

This module intentionally follows the MATLAB class method-by-method.  SciPy
is used only as the numerical backend for MATLAB's `intlinprog` models;
constraints, objectives, processing order, fallbacks, and post-processing
are kept from the MATLAB source.

The public result keys retain the existing Stage 2/3 API-facing convention.
Coordinates retain the MATLAB implementation's [X, Y] image-space meaning
(one-based pixel coordinates).
"""

import numpy as np
import cv2
from scipy import ndimage, sparse
from scipy.optimize import Bounds, LinearConstraint, milp


def _matlab_round(value):
    """MATLAB round for the non-negative image coordinates used here."""
    return int(np.floor(float(value) + 0.5))


class NodePlacerOptimizer:
    """NodePlacerOptimizer translated from the supplied MATLAB class."""

    def __init__(self):
        # MATLAB properties, in the same order and with the same defaults.
        self.DeadZoneThreshold_dBm = -60
        self.MinClusterAreaPx = 20
        self.ImageScaleFactor = 1.0
        self.RadiiToTest = np.arange(20, 121, 10)
        self.MaxCandidates = 300
        self.CoverageRatio_target = 0.80
        self.NodeCoverageRadius = 50
        self.NumSectors = 12
        self.MinDistPruning = 70
        self.SectorAngleDelta = 5
        self.StdDeviationThreshold = 10
        self.StrongSignalThreshold = -18
        self.MaxNodes = 0
        # v41: pure logging only, added alongside the existing radius sweep
        # in generateCandidates() below -- does not read from or feed back
        # into any computation, so the set-cover / ILP logic and math model
        # are byte-for-byte unchanged. Records every (radius, estNodes,
        # coverage%) triple the sweep already computes internally, purely
        # so it can be plotted afterward (3D trade-off landscape).
        self.radius_sweep_log = []

    # =====================================================================
    # Public methods
    # =====================================================================

    def run(self, campusImg, stage1_out, binaryMask):
        """Port of MATLAB `run`."""
        mag_db = np.asarray(stage1_out["mag_db"])
        src_x = stage1_out["src_x"]
        src_y = stage1_out["src_y"]

        if src_x is None or src_y is None:
            raise ValueError(
                f"Transmitter position is None: src_x={src_x}, src_y={src_y}. "
                "Ensure Stage 1 completed successfully before running Stage 2."
            )

        h_heat, w_heat = mag_db.shape

        # MATLAB: img = imresize(campusImg, ImageScaleFactor).
        img = np.asarray(campusImg)
        if self.ImageScaleFactor != 1.0:
            h_img = _matlab_round(img.shape[0] * self.ImageScaleFactor)
            w_img = _matlab_round(img.shape[1] * self.ImageScaleFactor)
            img = cv2.resize(img, (w_img, h_img), interpolation=cv2.INTER_LINEAR)
        h_img, w_img = img.shape[:2]

        # MATLAB maps the Stage 1 source coordinates into image space.
        tx = _matlab_round((src_x / w_heat) * w_img)
        ty = _matlab_round((src_y / h_heat) * h_img)
        tx = max(1, min(w_img, tx))
        ty = max(1, min(h_img, ty))

        # MATLAB: supplied mask is resized to image dimensions and thresholded.
        if binaryMask is not None and np.size(binaryMask) != 0:
            mask = np.asarray(binaryMask)
            if mask.shape != (h_img, w_img):
                mask = cv2.resize(
                    mask.astype(np.float32),
                    (w_img, h_img),
                    interpolation=cv2.INTER_CUBIC,
                )
            buildMask = mask > 0.5
        else:
            # This branch mirrors MATLAB's automatic mask path as closely as
            # possible.  Production Stage 2 always supplies the persisted ROI
            # mask, so this is retained for API compatibility.
            rgb = img[:, :, :3]
            gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
            threshold = _graythresh(gray) * 255.0 * 0.8
            buildMask = gray > threshold
        pathsMask = ~buildMask

        # MATLAB resizes the heatmap once to image space.
        if mag_db.shape != (h_img, w_img):
            heatImg = cv2.resize(
                mag_db.astype(np.float32),
                (w_img, h_img),
                interpolation=cv2.INTER_CUBIC,
            )
        else:
            heatImg = mag_db.copy()

        # Step 1: dead-zone extraction and clustering.
        clusters, numClusters, deadZoneMask = self.extractClusters(heatImg)
        if numClusters == 0:
            return self.emptyResult(
                img, [tx, ty], heatImg, buildMask,
                np.zeros((h_img, w_img), dtype=bool),
            )

        # Step 2: polar transformation.
        clusters = self.polarTransform(clusters, tx, ty)

        # Step 3: candidate location generation.
        candidates_xy, bestR, fullCandidates = self.generateCandidates(
            pathsMask, clusters, tx, ty, h_img, w_img
        )
        if len(candidates_xy) == 0:
            return self.emptyResult(
                img, [tx, ty], heatImg, buildMask, deadZoneMask
            )

        # Step 4: sector mapping.
        clusterSectors, activeSectors = self.mapSectors(clusters)

        # Step 5: adjacency.
        A_mat = self.buildAdjacency(clusters, candidates_xy)

        # Step 6: solve exactly the MATLAB branch selected by MaxNodes.
        if self.MaxNodes > 0:
            selectedIdx, covFracILP = self.solveBudgetILP(
                A_mat, candidates_xy, clusters, tx, ty, self.MaxNodes
            )
        else:
            covFracILP = None
            selectedIdx = self.solveMinCoverILP(
                A_mat, candidates_xy, clusters,
                clusterSectors, activeSectors, tx, ty
            )
        placedNodes = candidates_xy[np.asarray(selectedIdx, dtype=int)] if len(selectedIdx) else np.zeros((0, 2))

        # Step 7: strict air and boundary filter.
        placedNodes = self.strictAirFilter(placedNodes, pathsMask, h_img, w_img)
        nodesBeforeOpt = placedNodes.copy()

        # Steps 8-9: only in unconstrained mode.
        if self.MaxNodes == 0:
            placedNodes = self.rfSectorPrune(
                placedNodes, heatImg, tx, ty, w_img, h_img
            )
            placedNodes = self.spatialDiversityEnforce(placedNodes)

        # MATLAB's budget branch applies this post-placement step.
        if self.MaxNodes > 0 and len(placedNodes):
            placedNodes = self.removZeroGainNodes(placedNodes, clusters)

        coverageFraction = self.computeCoverage(placedNodes, clusters)

        return {
            "img": img,
            "transmitter": [tx, ty],
            "bestR": bestR,
            "heatImg": heatImg,
            "binaryMask": buildMask,
            "deadZoneMask": deadZoneMask,
            "candidateLocations": fullCandidates,
            "clusterCentroids": np.array(
                [[c["cx"], c["cy"]] for c in clusters], dtype=float
            ),
            "adjacency": A_mat,
            "nodesBeforeOpt": nodesBeforeOpt,
            "finalNodes": placedNodes,
            "coverageFraction": coverageFraction,
            # Retained for callers that used the previous internal result.
            "mag_db": heatImg,
            "covFracILP": covFracILP,
        }

    def runBudget(self, mainResult, maxNodes):
        """Port of MATLAB `runBudget` using fixed Stage 2 node positions."""
        finalNodes = _xy_array(mainResult.get("finalNodes", []))
        n = finalNodes.shape[0]

        if n == 0:
            budgetResult = dict(mainResult)
            budgetResult["finalNodes"] = np.zeros((0, 2))
            budgetResult["coverageFraction"] = 0
            return budgetResult

        if maxNodes >= n:
            return dict(mainResult)

        cc = _xy_array(mainResult.get("clusterCentroids", []))
        clusters = [
            {
                "cx": float(row[0]),
                "cy": float(row[1]),
                "area": 1,
                "meanRSSI": 0,
                "r": 0,
                "theta": 0,
            }
            for row in cc
        ]

        A_mat = self.buildAdjacency(clusters, finalNodes)
        selectedNodes = self.backwardElimination(
            finalNodes, A_mat, len(clusters), maxNodes
        )
        covFrac = self.computeCoverage(selectedNodes, clusters)

        budgetResult = dict(mainResult)
        budgetResult["finalNodes"] = selectedNodes
        budgetResult["nodesBeforeOpt"] = selectedNodes
        budgetResult["coverageFraction"] = covFrac
        return budgetResult

    # =====================================================================
    # MATLAB private methods: steps 1-5
    # =====================================================================

    def extractClusters(self, heatImg):
        """Port of MATLAB `extractClusters` and `bwconncomp(..., 8)`."""
        deadZoneMask = np.asarray(heatImg) < self.DeadZoneThreshold_dBm
        connectivity = np.ones((3, 3), dtype=np.uint8)
        labels, count = ndimage.label(deadZoneMask, structure=connectivity)

        # bwconncomp scans MATLAB arrays in column-major order.  scipy labels
        # components while scanning row-major, so reorder the component IDs by
        # their first MATLAB-linear pixel before building regionprops-like
        # records.
        component_order = []
        for k in range(1, count + 1):
            pixel_rows, pixel_cols = np.where(labels == k)
            if pixel_rows.size:
                first_linear = int(np.min(
                    pixel_rows + pixel_cols * heatImg.shape[0]
                ))
                component_order.append((first_linear, k))
        component_order.sort()

        clusters = []
        for _, k in component_order:
            pixel_idx = np.flatnonzero(labels == k)
            area = int(pixel_idx.size)
            if area < self.MinClusterAreaPx:
                continue

            # MATLAB regionprops Centroid is one-based [X,Y].  The array
            # indices are recovered in MATLAB's column-major order.
            rows0, cols0 = np.where(labels == k)
            clusters.append({
                "cx": float(cols0.mean() + 1),
                "cy": float(rows0.mean() + 1),
                "area": area,
                "meanRSSI": float(np.mean(np.asarray(heatImg)[rows0, cols0])),
                "r": 0.0,
                "theta": 0.0,
            })
        return clusters, len(clusters), deadZoneMask

    def polarTransform(self, clusters, tx, ty):
        """Port of MATLAB `polarTransform`."""
        for cluster in clusters:
            dx = cluster["cx"] - tx
            dy = cluster["cy"] - ty
            cluster["r"] = float(np.sqrt(dx * dx + dy * dy))
            cluster["theta"] = float(np.mod(np.degrees(np.arctan2(dy, dx)), 360))
        return clusters

    def generateCandidates(self, pathsMask, clusters, tx, ty, h_img, w_img):
        """Port of MATLAB `generateCandidates`."""
        margin = max(5, _matlab_round(self.NodeCoverageRadius / 2))
        edgeMask = np.zeros((h_img, w_img), dtype=bool)
        # MATLAB: margin+1:h_img-margin, margin+1:w_img-margin.
        if h_img - margin >= margin + 1 and w_img - margin >= margin + 1:
            edgeMask[margin:h_img - margin, margin:w_img - margin] = True
        interiorMask = np.asarray(pathsMask, dtype=bool) & edgeMask

        # MATLAB find() is column-major and returns one-based row/column.
        pixel_idx = np.flatnonzero(interiorMask.ravel(order="F"))
        rows0, cols0 = np.unravel_index(pixel_idx, (h_img, w_img), order="F")
        L_pool = np.column_stack((cols0 + 1, rows0 + 1)).astype(float)

        if L_pool.shape[0] == 0:
            pixel_idx = np.flatnonzero(np.asarray(pathsMask).ravel(order="F"))
            rows0, cols0 = np.unravel_index(pixel_idx, (h_img, w_img), order="F")
            L_pool = np.column_stack((cols0 + 1, rows0 + 1)).astype(float)
        if L_pool.shape[0] == 0:
            L_pool = np.array([[_matlab_round(w_img / 2), _matlab_round(h_img / 2)]], dtype=float)

        fullCandidates = L_pool.copy()
        D_xy = np.array([[c["cx"], c["cy"]] for c in clusters], dtype=float)
        k = len(clusters)
        bestR = int(self.RadiiToTest[0])
        minNodes = np.inf

        for radius in self.RadiiToTest:
            distTx = np.sqrt(
                (L_pool[:, 0] - tx) ** 2 + (L_pool[:, 1] - ty) ** 2
            )
            validPool = L_pool[distTx >= radius, :]
            if validPool.shape[0] == 0:
                continue

            step = max(1, int(np.floor(validPool.shape[0] / 120)))
            L_sub = validPool[::step, :]

            A_temp = np.zeros((k, L_sub.shape[0]), dtype=bool)
            for j in range(L_sub.shape[0]):
                d = np.sqrt(
                    (D_xy[:, 0] - L_sub[j, 0]) ** 2
                    + (D_xy[:, 1] - L_sub[j, 1]) ** 2
                )
                A_temp[d <= self.NodeCoverageRadius, j] = True

            covRatio = np.sum(np.any(A_temp, axis=1)) / k
            estNodes = int(np.sum(np.any(A_temp, axis=0)))
            # v41: log only -- same values the line below already computed,
            # just captured for the 3D trade-off plot. No effect on bestR
            # selection or any downstream placement logic.
            self.radius_sweep_log.append(
                (float(radius), estNodes, covRatio * 100.0))
            if covRatio >= self.CoverageRatio_target and estNodes < minNodes:
                minNodes = estNodes
                bestR = int(radius)

        distTx = np.sqrt(
            (L_pool[:, 0] - tx) ** 2 + (L_pool[:, 1] - ty) ** 2
        )
        finalPool = L_pool[distTx >= bestR, :]
        if finalPool.shape[0] == 0:
            finalPool = L_pool

        step = max(1, int(np.floor(finalPool.shape[0] / self.MaxCandidates)))
        candidates_xy = finalPool[::step, :]
        return candidates_xy, bestR, fullCandidates

    def mapSectors(self, clusters):
        """Port of MATLAB `mapSectors`."""
        deltaTh = 360 / self.NumSectors
        clusterSectors = np.zeros(len(clusters), dtype=int)
        for k, cluster in enumerate(clusters):
            clusterSectors[k] = min(
                int(np.floor(cluster["theta"] / deltaTh)) + 1,
                self.NumSectors,
            )
        activeSectors = np.unique(clusterSectors)
        return clusterSectors, activeSectors

    def buildAdjacency(self, clusters, candidates_xy):
        """Port of MATLAB `buildAdjacency`."""
        candidates_xy = _xy_array(candidates_xy)
        adjacency = np.zeros((len(clusters), candidates_xy.shape[0]), dtype=bool)
        for j in range(candidates_xy.shape[0]):
            dx = np.array([c["cx"] for c in clusters]) - candidates_xy[j, 0]
            dy = np.array([c["cy"] for c in clusters]) - candidates_xy[j, 1]
            adjacency[:, j] = np.sqrt(dx * dx + dy * dy) <= self.NodeCoverageRadius
        return adjacency

    # =====================================================================
    # MATLAB private methods: step 6
    # =====================================================================

    def solveBudgetILP(self, A_mat, candidates_xy, clusters, tx, ty, maxNodes):
        """Port of MATLAB `solveBudgetILP`."""
        A_mat = np.asarray(A_mat, dtype=bool)
        candidates_xy = _xy_array(candidates_xy)
        k, m = A_mat.shape
        tau = self.NodeCoverageRadius
        nVars = m + k

        # C1: sum(x) <= maxNodes.
        rows = [sparse.csr_matrix(np.r_[np.ones(m), np.zeros(k)][None, :])]
        lower = [-np.inf]
        upper = [maxNodes]

        # C2: [-A | I] z <= 0.
        rows.append(sparse.hstack([-sparse.csr_matrix(A_mat.astype(float)), sparse.eye(k)]))
        lower.extend([-np.inf] * k)
        upper.extend([0.0] * k)

        # C3: x_j + x_k <= 1 for every pair with distance < tau.
        dx_pair = candidates_xy[:, 0, None] - candidates_xy[None, :, 0]
        dy_pair = candidates_xy[:, 1, None] - candidates_xy[None, :, 1]
        pairD = np.sqrt(dx_pair * dx_pair + dy_pair * dy_pair)
        jIdx, kIdx = np.where(np.triu(pairD < tau, 1))
        if len(jIdx):
            pair_rows = np.arange(len(jIdx))
            A_c3 = sparse.coo_matrix(
                (
                    np.ones(len(jIdx) * 2),
                    (
                        np.repeat(pair_rows, 2),
                        np.column_stack((jIdx, kIdx)).ravel(),
                    ),
                ),
                shape=(len(jIdx), nVars),
            ).tocsr()
            rows.append(A_c3)
            lower.extend([-np.inf] * len(jIdx))
            upper.extend([1.0] * len(jIdx))

        # C4: candidate-sector anti-clustering.
        k_sec = self.NumSectors
        maxPerSec = int(np.ceil(maxNodes / k_sec))
        deltaTh = 360 / k_sec
        candAngles = np.mod(
            np.degrees(np.arctan2(candidates_xy[:, 1] - ty, candidates_xy[:, 0] - tx)),
            360,
        )
        c4_rows = []
        for s in range(1, k_sec + 1):
            aMin = (s - 1) * deltaTh
            aMax = s * deltaTh
            inSec = np.flatnonzero((candAngles >= aMin) & (candAngles < aMax))
            if len(inSec) > maxPerSec:
                row = np.zeros(nVars)
                row[inSec] = 1
                c4_rows.append(row)
        if c4_rows:
            rows.append(sparse.csr_matrix(np.asarray(c4_rows)))
            lower.extend([-np.inf] * len(c4_rows))
            upper.extend([float(maxPerSec)] * len(c4_rows))

        A_ineq = sparse.vstack(rows, format="csr")
        f = np.r_[np.zeros(m), -np.ones(k)]
        result = milp(
            c=f,
            integrality=np.ones(nVars),
            bounds=Bounds(np.zeros(nVars), np.ones(nVars)),
            constraints=LinearConstraint(
                A_ineq, np.asarray(lower), np.asarray(upper)
            ),
            options={"time_limit": 120},
        )

        if result.success and result.x is not None:
            selectedIdx = np.flatnonzero(result.x[:m] > 0.5)
            covFrac = float(np.sum(result.x[m:] > 0.5) / k) if k else 0
        else:
            selectedIdx = self.greedyBudgetSpread(
                A_mat, candidates_xy, maxNodes, tau
            )
            covFrac = self.computeCoverage(
                candidates_xy[selectedIdx, :] if len(selectedIdx) else np.zeros((0, 2)),
                clusters,
            )

        if len(selectedIdx) == 0:
            selectedIdx = np.arange(min(maxNodes, m), dtype=int)
            covFrac = 0
        return selectedIdx, covFrac

    def solveMinCoverILP(
        self, A_mat, candidates_xy, clusters, clusterSectors, activeSectors, tx, ty
    ):
        """Port of MATLAB `solveMinCoverILP`."""
        A_mat = np.asarray(A_mat, dtype=bool)
        candidates_xy = _xy_array(candidates_xy)
        m = A_mat.shape[1]

        rowsCovered = np.flatnonzero(np.any(A_mat, axis=1))
        A_c1 = sparse.csr_matrix(A_mat[rowsCovered].astype(float))
        lower = np.ones(len(rowsCovered))
        upper = np.full(len(rowsCovered), np.inf)

        deltaTh = 360 / self.NumSectors
        candAngles = np.mod(
            np.degrees(np.arctan2(candidates_xy[:, 1] - ty, candidates_xy[:, 0] - tx)),
            360,
        )
        candSec = np.minimum(
            np.floor(candAngles / deltaTh).astype(int) + 1,
            self.NumSectors,
        )

        b_sec = []
        for sector in np.asarray(activeSectors, dtype=int):
            b_sec.append((candSec == sector).astype(float))
        if b_sec:
            B_sec = np.asarray(b_sec)
            validSec = np.any(B_sec, axis=1)
            B_sec_valid = sparse.csr_matrix(B_sec[validSec])
            rows = sparse.vstack([A_c1, B_sec_valid], format="csr")
            lower = np.r_[lower, np.ones(int(np.sum(validSec)))]
            upper = np.full(rows.shape[0], np.inf)
        else:
            rows = A_c1

        result = milp(
            c=np.ones(m),
            integrality=np.ones(m),
            bounds=Bounds(np.zeros(m), np.ones(m)),
            constraints=LinearConstraint(rows, lower, upper),
            options={"time_limit": 120},
        )
        if result.success and result.x is not None:
            selectedIdx = np.flatnonzero(result.x > 0.5)
        else:
            selectedIdx = self.greedySetCover(A_mat)
        if len(selectedIdx) == 0:
            selectedIdx = np.arange(min(10, m), dtype=int)
        return selectedIdx

    def backwardElimination(self, finalNodes, A_mat, K, maxNodes):
        """Port of MATLAB `backwardElimination`."""
        finalNodes = _xy_array(finalNodes)
        n = finalNodes.shape[0]
        if maxNodes >= n:
            return finalNodes

        active = np.ones(n, dtype=bool)
        while int(np.sum(active)) > maxNodes:
            activeIdx = np.flatnonzero(active)
            covNow = int(np.sum(np.any(A_mat[:, active], axis=1))) if K else 0
            minLoss = np.inf
            dropNode = int(activeIdx[0])

            for j in activeIdx:
                active[j] = False
                covWithout = int(np.sum(np.any(A_mat[:, active], axis=1))) if K else 0
                active[j] = True
                loss = covNow - covWithout
                if loss < minLoss:
                    minLoss = loss
                    dropNode = int(j)
            active[dropNode] = False
        return finalNodes[active, :]

    def selectBestSubsetILP(self, A_mat, N, K, maxNodes):
        """MATLAB helper retained for compatibility; runBudget uses backward elimination."""
        A_mat = np.asarray(A_mat, dtype=bool)
        nVars = N + K
        rows = [
            sparse.csr_matrix(np.r_[np.ones(N), np.zeros(K)][None, :]),
            sparse.hstack([-sparse.csr_matrix(A_mat.astype(float)), sparse.eye(K)]),
        ]
        lower = [-np.inf] + [-np.inf] * K
        upper = [maxNodes] + [0.0] * K
        result = milp(
            c=np.r_[np.zeros(N), -np.ones(K)],
            integrality=np.ones(nVars),
            bounds=Bounds(np.zeros(nVars), np.ones(nVars)),
            constraints=LinearConstraint(
                sparse.vstack(rows, format="csr"),
                np.asarray(lower),
                np.asarray(upper),
            ),
            options={"time_limit": 60},
        )
        if result.success and result.x is not None:
            selectedIdx = np.flatnonzero(result.x[:N] > 0.5)
            covFrac = float(np.sum(result.x[N:] > 0.5) / K) if K else 0
        else:
            uncovered = np.ones(K, dtype=bool)
            selected = []
            for _ in range(maxNodes):
                gains = np.sum(A_mat[uncovered, :], axis=0)
                if np.all(gains == 0):
                    break
                best = int(np.argmax(gains))
                selected.append(best)
                uncovered[A_mat[:, best]] = False
                if not np.any(uncovered):
                    break
            selectedIdx = np.asarray(selected, dtype=int)
            covFrac = float(np.sum(~uncovered) / K) if K else 0
        if len(selectedIdx) == 0:
            selectedIdx = np.arange(min(maxNodes, N), dtype=int)
            covFrac = 0
        return selectedIdx, covFrac

    def greedyBudgetSpread(self, A_mat, candidates_xy, maxNodes, tau):
        """Port of MATLAB `greedyBudgetSpread`."""
        A_mat = np.asarray(A_mat, dtype=bool)
        candidates_xy = _xy_array(candidates_xy)
        m = candidates_xy.shape[0]
        k = A_mat.shape[0]
        uncovered = np.ones(k, dtype=bool)
        selectedIdx = np.zeros(maxNodes, dtype=int)
        nPlaced = 0
        placed_xy = np.zeros((maxNodes, 2), dtype=float)

        for _ in range(maxNodes):
            if nPlaced == 0:
                validMask = np.ones(m, dtype=bool)
            else:
                dmat = np.sqrt(
                    (candidates_xy[:, 0, None] - placed_xy[:nPlaced, 0][None, :]) ** 2
                    + (candidates_xy[:, 1, None] - placed_xy[:nPlaced, 1][None, :]) ** 2
                )
                validMask = np.min(dmat, axis=1) >= tau

            relaxed = tau
            while not np.any(validMask) and relaxed > tau * 0.25:
                relaxed *= 0.75
                dmat = np.sqrt(
                    (candidates_xy[:, 0, None] - placed_xy[:nPlaced, 0][None, :]) ** 2
                    + (candidates_xy[:, 1, None] - placed_xy[:nPlaced, 1][None, :]) ** 2
                )
                validMask = np.min(dmat, axis=1) >= relaxed
            if not np.any(validMask):
                break

            gains = np.zeros(m)
            if np.any(uncovered):
                gains[validMask] = np.sum(A_mat[uncovered][:, validMask], axis=0)

            if np.all(gains[validMask] == 0):
                if nPlaced == 0:
                    scores = np.sum(A_mat, axis=0).astype(float)
                    scores[~validMask] = -np.inf
                else:
                    scores = np.min(dmat, axis=1)
                    scores[~validMask] = -np.inf
                best = int(np.argmax(scores))
            else:
                best = int(np.argmax(gains))

            selectedIdx[nPlaced] = best
            placed_xy[nPlaced, :] = candidates_xy[best, :]
            nPlaced += 1
            uncovered[A_mat[:, best]] = False
            if not np.any(uncovered):
                break

        return np.unique(selectedIdx[:nPlaced])

    def greedySetCover(self, A_mat):
        """Port of MATLAB `greedySetCover`."""
        A_mat = np.asarray(A_mat, dtype=bool)
        uncovered = np.ones(A_mat.shape[0], dtype=bool)
        selectedIdx = []
        while np.any(uncovered):
            gains = np.sum(A_mat[uncovered, :], axis=0)
            if np.all(gains == 0):
                break
            best = int(np.argmax(gains))
            selectedIdx.append(best)
            uncovered[A_mat[:, best]] = False
        return np.asarray(selectedIdx, dtype=int)

    # =====================================================================
    # MATLAB private methods: steps 7-9 and helpers
    # =====================================================================

    def strictAirFilter(self, nodes, pathsMask, h_img, w_img):
        """Port of MATLAB `strictAirFilter`."""
        nodes = _xy_array(nodes)
        margin = max(5, _matlab_round(self.NodeCoverageRadius / 2))
        keep = np.zeros(nodes.shape[0], dtype=bool)
        for i, node in enumerate(nodes):
            c = min(w_img, max(1, _matlab_round(node[0])))
            r = min(h_img, max(1, _matlab_round(node[1])))
            inMask = bool(pathsMask[r - 1, c - 1])
            inBounds = (
                c > margin and c < (w_img - margin)
                and r > margin and r < (h_img - margin)
            )
            if inMask and inBounds:
                keep[i] = True
        # MATLAB only replaces the input if at least one node survives.
        return nodes[keep, :] if np.any(keep) else nodes

    def rfSectorPrune(self, nodes, heatImg, tx, ty, w_img, h_img):
        """Port of MATLAB `rfSectorPrune`."""
        nodes = _xy_array(nodes)
        if nodes.shape[0] == 0:
            return nodes

        x_grid, y_grid = np.meshgrid(
            np.arange(1, w_img + 1), np.arange(1, h_img + 1)
        )
        pixAngles = np.mod(np.degrees(np.arctan2(y_grid - ty, x_grid - tx)), 360)
        nodeAngles = np.mod(
            np.degrees(np.arctan2(nodes[:, 1] - ty, nodes[:, 0] - tx)), 360
        )
        keepNode = np.ones(nodes.shape[0], dtype=bool)
        for aS in np.arange(0, 360, self.SectorAngleDelta):
            aE = aS + self.SectorAngleDelta
            inSec = np.flatnonzero((nodeAngles >= aS) & (nodeAngles < aE))
            if len(inSec) == 0:
                continue
            sectorVals = heatImg[(pixAngles >= aS) & (pixAngles < aE)]
            sectorStd = np.std(sectorVals, ddof=1) if sectorVals.size > 1 else np.nan
            if sectorStd > self.StdDeviationThreshold:
                for ni in inSec:
                    nx = min(w_img, max(1, _matlab_round(nodes[ni, 0])))
                    ny = min(h_img, max(1, _matlab_round(nodes[ni, 1])))
                    if heatImg[ny - 1, nx - 1] > self.StrongSignalThreshold:
                        keepNode[ni] = False
        filtered = nodes[keepNode, :]
        return filtered if filtered.shape[0] else nodes

    def spatialDiversityEnforce(self, nodes):
        """Port of MATLAB `spatialDiversityEnforce`."""
        nodes = _xy_array(nodes)
        if nodes.shape[0] == 0:
            return nodes
        kept = [nodes[0].copy()]
        for i in range(1, nodes.shape[0]):
            distances = np.sqrt(np.sum((np.asarray(kept) - nodes[i]) ** 2, axis=1))
            if np.all(distances > self.MinDistPruning):
                kept.append(nodes[i].copy())
        return np.asarray(kept)

    def removZeroGainNodes(self, nodes, clusters):
        """Port of MATLAB's intentionally named `removZeroGainNodes`."""
        nodes = _xy_array(nodes)
        if nodes.shape[0] == 0 or len(clusters) == 0:
            return nodes
        n = nodes.shape[0]
        keep = np.ones(n, dtype=bool)
        cx = np.array([c["cx"] for c in clusters])
        cy = np.array([c["cy"] for c in clusters])

        for i in range(n):
            others = nodes[np.arange(n) != i, :]
            if others.shape[0] == 0:
                keep[i] = True
                continue
            di = np.sqrt((cx - nodes[i, 0]) ** 2 + (cy - nodes[i, 1]) ** 2)
            myClusters = di <= self.NodeCoverageRadius
            if not np.any(myClusters):
                keep[i] = False
                continue
            myCx = cx[myClusters]
            myCy = cy[myClusters]
            covByOther = np.zeros(myCx.shape[0], dtype=bool)
            for other in others:
                do = np.sqrt((myCx - other[0]) ** 2 + (myCy - other[1]) ** 2)
                covByOther[do <= self.NodeCoverageRadius] = True
            if np.all(covByOther):
                keep[i] = False
        return nodes[keep, :] if np.any(keep) else nodes

    def computeCoverage(self, nodes, clusters):
        """Port of MATLAB `computeCoverage`."""
        nodes = _xy_array(nodes)
        k = len(clusters)
        if k == 0 or nodes.shape[0] == 0:
            return 0
        cx = np.array([c["cx"] for c in clusters])
        cy = np.array([c["cy"] for c in clusters])
        covered = np.zeros(k, dtype=bool)
        for node in nodes:
            d = np.sqrt((cx - node[0]) ** 2 + (cy - node[1]) ** 2)
            covered[d <= self.NodeCoverageRadius] = True
        return float(np.sum(covered) / k)

    def emptyResult(self, img, transmitter, heatImg, buildMask, deadZoneMask):
        """Port of MATLAB `emptyResult`."""
        return {
            "img": img,
            "transmitter": transmitter,
            "bestR": 0,
            "heatImg": heatImg,
            "binaryMask": buildMask,
            "deadZoneMask": deadZoneMask,
            "candidateLocations": np.zeros((0, 2)),
            "clusterCentroids": np.zeros((0, 2)),
            "adjacency": np.zeros((0, 0), dtype=bool),
            "nodesBeforeOpt": np.zeros((0, 2)),
            "finalNodes": np.zeros((0, 2)),
            "coverageFraction": 0,
            "mag_db": heatImg,
        }


def _xy_array(value):
    arr = np.asarray(value, dtype=float)
    if arr.size == 0:
        return np.zeros((0, 2), dtype=float)
    return arr.reshape((-1, 2))


def _graythresh(gray):
    """Small direct implementation of MATLAB graythresh/Otsu threshold."""
    values = np.asarray(gray, dtype=np.uint8).ravel()
    hist = np.bincount(values, minlength=256).astype(float)
    probability = hist / max(1, values.size)
    omega = np.cumsum(probability)
    mu = np.cumsum(probability * np.arange(256))
    mu_t = mu[-1]
    denominator = omega * (1 - omega)
    numerator = (mu_t * omega - mu) ** 2
    sigma = np.divide(
        numerator, denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0,
    )
    return float(np.argmax(sigma) / 255.0)

print("✅  NodePlacerOptimizer ready (faithful port: sector clustering + "
      "polar transform + ILP set-cover, same logic/math as reference).")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 6 — USER INPUTS  (input() prompts, validation, defaults)         ║
# ║                                                                          ║
# ║   CHANGED: only ONE image is uploaded now (the floorplan). The binary   ║
# ║   building mask is generated internally by thresholding the floorplan   ║
# ║   itself (same dark-pixel-=-building rule the old mask.png upload used) ║
# ║   — no separate mask.png upload/step needed anymore.                    ║
# ║   REMOVED: "pml_thickness" prompt — it was collected but never actually ║
# ║   used anywhere in the solver (the sponge/absorbing layer is a fixed    ║
# ║   SPONGE_THICKNESS in pixels, unrelated to this input); asking for it   ║
# ║   implied it did something when it didn't, so it's gone.                ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# v23: single global switch -- asked ONCE, right at the top of the whole
# input flow. If the user picks "defaults", every _prompt_float/_prompt_int
# call below silently returns its own default with no input() prompt at
# all; the frequency/material/output-dir prompts (which don't go through
# those two helpers) check the same flag directly. Nothing about the
# validators/defaults themselves changed -- this only decides whether the
# user is asked or the default is taken automatically.
_USE_DEFAULTS = False


def _ask_use_defaults():
    global _USE_DEFAULTS
    print("=" * 72)
    print("  5G PHASED ARRAY COVERAGE SIMULATOR -- PARAMETER SETUP")
    print("=" * 72)
    raw = input("Use default parameters for everything, or set your own? "
                 "[d]efaults / [c]ustom [d]: ").strip().lower()
    _USE_DEFAULTS = raw in ("", "d", "default", "defaults")
    if _USE_DEFAULTS:
        print("  ✓ Using default parameters everywhere (you'll still be asked "
              "for the map/frequency-unit confirmation if needed).")
    else:
        print("  ✓ Custom mode -- you'll be prompted for each parameter "
              "(blank = default).")
    return _USE_DEFAULTS


def _prompt_float(prompt, default, validator=None, error_msg=None):
    """Prompt for a float; blank input uses default. Repeats on invalid input.
    In defaults mode, returns `default` immediately with no input() call."""
    if _USE_DEFAULTS:
        return default
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        try:
            val = default if raw == "" else float(raw)
        except ValueError:
            print(f"  ❌ Invalid number. Try again.")
            continue
        if validator and not validator(val):
            print(f"  ❌ {error_msg or 'Value out of range.'} Try again.")
            continue
        return val


def _prompt_int(prompt, default, validator=None, error_msg=None):
    if _USE_DEFAULTS:
        return default
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        try:
            val = default if raw == "" else int(raw)
        except ValueError:
            print(f"  ❌ Invalid integer. Try again.")
            continue
        if validator and not validator(val):
            print(f"  ❌ {error_msg or 'Value out of range.'} Try again.")
            continue
        return val


class _AmbiguousFrequency(Exception):
    """Raised when a bare (unsuffixed) number could plausibly mean either
    GHz or MHz -- forces an explicit re-prompt instead of silently
    guessing, since guessing wrong is a 1000x physics error."""
    def __init__(self, as_ghz, as_mhz):
        self.as_ghz = as_ghz
        self.as_mhz = as_mhz


def _parse_frequency(raw, default_hz=50e6):
    """'2.4GHz' -> 2.4e9; '915MHz' -> 915e6; '50MHz' -> 50e6.
    A bare number with no suffix is never silently guessed."""
    raw = raw.strip()
    if raw == "":
        return default_hz
    s = raw.upper().replace(" ", "")
    if s.endswith("GHZ"):
        return float(s[:-3]) * 1e9
    if s.endswith("MHZ"):
        return float(s[:-3]) * 1e6
    if s.endswith("KHZ"):
        return float(s[:-3]) * 1e3
    if s.endswith("HZ"):
        return float(s[:-2])

    val = float(s)
    raise _AmbiguousFrequency(as_ghz=val * 1e9, as_mhz=val * 1e6)


def collect_user_inputs_part1():
    """
    Everything that does NOT depend on the map's real-world width/height.
    v21: dimensions are no longer asked here at all -- matching the
    reference site exactly, which never prompts for width/height either;
    it silently falls back to a 0.5 m/px cell size. Width/height are now
    auto-derived from the uploaded image's own pixel size AFTER upload
    (see upload_map_and_generate_mask / collect_user_inputs_part2), the
    same fallback the reference uses.
    """
    _ask_use_defaults()
    print("=" * 72)
    print("  5G PHASED ARRAY COVERAGE SIMULATOR — REAL-TIME BEAM STEERING")
    print("=" * 72)

    cfg = {}

    print("\n─── Frequency ───")
    while True:
        raw = "" if _USE_DEFAULTS else \
            input("Frequency -- please include units, e.g. 50MHz, 2.4GHz, 915MHz [50MHz]: ").strip()
        try:
            cfg["freq_hz"] = _parse_frequency(raw, 50e6)
            if cfg["freq_hz"] <= 0:
                print("  ❌ Frequency must be positive.")
                continue
            print(f"  ✓ Parsed as {cfg['freq_hz']/1e9:.4f} GHz "
                  f"({cfg['freq_hz']/1e6:.1f} MHz).")
            break
        except _AmbiguousFrequency as amb:
            print(f"  ⚠️  '{raw}' has no unit. Did you mean "
                  f"{amb.as_ghz/1e9:.3g} GHz or {amb.as_mhz/1e6:.3g} MHz?")
            unit = input("  Type 'ghz' or 'mhz' to confirm: ").strip().lower()
            if unit in ("g", "ghz"):
                cfg["freq_hz"] = amb.as_ghz
                print(f"  ✓ Using {cfg['freq_hz']/1e9:.4f} GHz.")
                break
            elif unit in ("m", "mhz"):
                cfg["freq_hz"] = amb.as_mhz
                print(f"  ✓ Using {cfg['freq_hz']/1e6:.4f} MHz.")
                break
            else:
                print("  ❌ Not recognized -- please re-enter the frequency with units.")
        except ValueError:
            print("  ❌ Could not parse frequency. Examples: 50MHz, 2.4GHz, 915MHz")

    print("\n─── Transmit Power ───")
    # v25 FIX: default reverted to 22.0 -- this MUST match the reference
    # engine's stage1_engine.py build_params() sourceValue default (22)
    # exactly, since source_amp is now used raw with no calibration step.
    # The previous v24 default of 48.0 was wrong (a leftover guess, not
    # derived from the reference) and was the direct cause of the peak
    # coming out around 48 dB instead of the reference's ~8.4 dB.
    # v28 REVERT: v27's 620 was wrong -- it was reverse-engineered to hit
    # an arbitrary +30 dB display target instead of matching the reference
    # engine. source_amp is a forcing term in a linear PDE solve (A*E=-b),
    # NOT a dB value you read back out directly -- mag_db=20*log10(|E|) is
    # a SOLVED quantity that depends on grid spacing h, k0, and how the
    # discrete Laplacian inverts locally near the source cell. There is no
    # valid amp value that makes peak dB literally equal the amp number.
    # The reference engine (wifi_sim_engine/stage1_engine.py) uses
    # sourceValue=22 and its own documented/verified output is "True
    # Peak: 8.4 dB" -- NOT 22, NOT 30. That 8.4 dB is the correct,
    # validated ground truth for amp=22 on this physics. Back to 22.0.
    # v41: user-facing input is now the TARGET PEAK dB they want to SEE on
    # the path-loss graph / heatmap, not a raw PDE source amplitude (that
    # raw number was confusing to read/set directly -- it has no simple
    # physical meaning on its own). The raw amplitude the solver actually
    # needs is still computed internally exactly as before -- nothing about
    # the physics changes -- it's just done via a calibration step in main()
    # AFTER the field is solved, using the exact linear relationship
    # mag_db = 20*log10(amp) + mag_db(unit amp): a uniform additive dB shift
    # is mathematically IDENTICAL to having solved with a different source
    # amplitude to begin with (see _rescale_to_target_peak in Cell 8),
    # so nothing is faked -- it's an exact algebraic rescale of real,
    # already-solved data. These two internal amplitudes are the SAME
    # fixed reference amplitudes the notebook always used (22.0 for the
    # single antenna, -5 dB per element for the array) -- they are only
    # a calibration starting point now, not something the user sets.
    # v41: renamed with a leading underscore -- these are INTERNAL solver
    # seeds only (the raw PDE source amplitude needed to run a solve at
    # all), never the number the user actually sees. They get calibrated
    # away by _rescale_to_target_peak() in main() right after solving, so
    # their literal value is meaningless on its own and is hidden from the
    # parameter summary printout below (which now skips underscore keys)
    # to stop it from looking like a second, unused "10 vs 22" setting.
    cfg["_single_antenna_probe_amp"] = 22.0
    cfg["_array_element_probe_amp"] = round(22.0 * 10 ** (-5 / 20), 3)

    cfg["target_peak_dbm_single"] = _prompt_float(
        "Desired peak signal strength to see on the SINGLE-ANTENNA map / "
        "path-loss graph (dB) -- e.g. enter 10 to see the path-loss curve "
        "start around +10 dB near the source",
        10.0, lambda v: -200 <= v <= 200,
        "Must be between -200 and 200")
    cfg["target_peak_dbm_array"] = _prompt_float(
        "Desired peak signal strength to see on the PHASED-ARRAY (max-hold "
        "total) map / path-loss graph (dB)",
        5.0, lambda v: -200 <= v <= 200,
        "Must be between -200 and 200")

    return cfg


def collect_user_inputs_part2(cfg):
    """
    Everything that DOES depend on the map's real-world width/height
    (antenna position bounds, node-placement radius defaults). Called
    AFTER upload_map_and_generate_mask() has set cfg["width"]/["height"]
    from the image's own pixel size at 0.5 m/px (see that function).
    """
    print("\n─── Antenna Position ───")
    cfg["antenna_x"] = _prompt_float(
        "Antenna X (m)", min(20.0, cfg["width"]),
        lambda v: 0 <= v <= cfg["width"], f"X must be between 0 and {cfg['width']}")
    cfg["antenna_y"] = _prompt_float(
        "Antenna Y (m)", min(25.0, cfg["height"]),
        lambda v: 0 <= v <= cfg["height"], f"Y must be between 0 and {cfg['height']}")

    print("\n─── Building Material ───")
    print("  1=Concrete  2=Glass  3=Brick  4=Reinforced  5=Custom")
    _MATERIALS = {
        "1": (3.5, 0.062, "Concrete"),
        "2": (6.0, 0.030, "Glass"),
        "3": (4.0, 0.080, "Brick"),
        "4": (8.0, 0.120, "Reinforced concrete"),
    }
    mat_choice = _prompt_int("Material [1-5]", 1, lambda v: 1 <= v <= 5,
                              "Choose 1-5")
    if str(mat_choice) in _MATERIALS:
        eps_r, sigma, mat_name = _MATERIALS[str(mat_choice)]
        print(f"  ✓ {mat_name}: εr={eps_r}, σ={sigma} S/m")
    else:
        eps_r = _prompt_float("  Custom εr", 3.5, lambda v: v >= 1,
                               "εr must be >= 1")
        sigma = _prompt_float("  Custom σ (S/m)", 0.01, lambda v: v > 0,
                               "σ must be > 0")
        mat_name = "Custom"
    cfg["eps_r"] = eps_r
    cfg["sigma"] = sigma
    cfg["material_name"] = mat_name
    cfg["alpha_air"] = 0.00002
    # v25 FIX: the _MATERIALS table's sigma column was already tuned to be
    # the alpha_eff value directly (Concrete=0.062 == reference's own
    # build_params() default absorptionCoeff exactly). Running it through
    # min(0.5, sigma*6.0 + 0.02*(eps_r-1)) double-converted it, e.g.
    # Concrete 0.062 -> 0.422 (~7x the reference's wall loss). That made
    # every building far more opaque than the reference, starving
    # anything behind a building and pushing dead-zone detection out to
    # only the farthest-corner pixels -- the edge-clustered node
    # placement reported.
    cfg["alpha_eff"] = sigma

    print("\n─── Grid Resolution ───")
    print("  Note: the simulation grid resolution is determined by your")
    print("  uploaded floorplan image's own pixel size. PPW here is only")
    print("  used as a target to check that resolution against, and to warn")
    print("  you if it's too coarse for the chosen frequency.")
    cfg["ppw_target"] = _prompt_int("Target PPW (points/wavelength)", 10,
                                     lambda v: v >= 5, "PPW must be >= 5")

    print("\n─── Node Placement & Optimization (NodePlacerOptimizer) ───")
    print("  Dead zones are defined as the weakest X percentile of the map's")
    print("  own RSSI values (not a fixed dBm cutoff) -- this reliably finds")
    print("  several weak-spot clusters instead of just one or zero.")
    cfg["dead_zone_percentile"] = _prompt_int(
        "Dead Zone Threshold (percentile)", 35, lambda v: 0 < v < 100,
        "Percentile must be between 0 and 100")
    # v19 fix: these are now METERS, not pixels. Pixel-based defaults broke
    # on any floorplan resolution other than the one they were tuned on --
    # e.g. a 1067x1474px map over 100x100m is 0.068 m/px, so the old
    # "50px" default meant a 3.4m real-world coverage radius (tiny,
    # produces dozens of crowded nodes); a lower-res map at 0.5 m/px made
    # the SAME "50px" mean 25m instead. Meters are resolution-independent;
    # converted to px internally (run_node_placement) using this map's
    # own h (m/px) once the grid exists.
    # Defaults scale with THIS map's own width (~8%/12%, matching the
    # proportions the reference site's fixed-px defaults produce on its
    # own grid) rather than a flat meter number -- a flat number looks
    # reasonable on a 100m map and wildly wrong on a 20m or 500m one.
    # width is already known at this point in the prompt flow.
    _default_cov_m = round(cfg["width"] * 0.08, 1)
    _default_prune_m = round(cfg["width"] * 0.12, 1)
    cfg["node_coverage_radius_m"] = _prompt_float(
        "Node Coverage Radius (m)", _default_cov_m, lambda v: v > 0,
        "Radius must be > 0")
    cfg["min_dist_pruning_m"] = _prompt_float(
        "Min Distance Pruning (m)", _default_prune_m, lambda v: v > 0,
        "Must be > 0")
    cfg["min_cluster_area"] = _prompt_int(
        "Min Cluster Area (px)", 10, lambda v: v > 0,
        "Must be > 0")
    cfg["max_candidates"] = _prompt_int(
        "Max Candidates", 300, lambda v: v > 0,
        "Must be > 0")
    cfg["max_nodes"] = _prompt_int(
        "Max Nodes (0 = unlimited)", 0, lambda v: v >= 0,
        "Must be >= 0")
    cfg["dbm_min"] = _prompt_float("Minimum dB (colorbar/display scale)", -90.0)
    cfg["dbm_max"] = _prompt_float("Maximum dB (colorbar/display scale)", 20.0)

    print("\n─── Target Eligibility ───")
    print("  NodePlacerOptimizer places nodes wherever it finds dead zones,")
    print("  but a placed node's own signal might still be decent. This")
    print("  threshold decides which placed nodes are weak enough to be")
    print("  worth steering the beam toward.")
    print("  Leave blank to auto-match the dead-zone percentile threshold")
    print("  above (recommended) -- a fixed dBm number here can silently")
    print("  disagree with that percentile and target zero nodes, ending")
    print("  the run right after node placement with nothing left to do.")
    _threshold_raw = "" if _USE_DEFAULTS else \
        input("Target RSSI threshold (dBm) [auto]: ").strip()
    cfg["target_rssi_threshold"] = float(_threshold_raw) if _threshold_raw else None

    print("\n─── Real-Time Loop ───")
    cfg["n_iterations"] = _prompt_int("Iterations N", 10, lambda v: v >= 1,
                                       "N must be >= 1")

    print("\n─── Output ───")
    out_dir = ("" if _USE_DEFAULTS else input("Output dir [./frames]: ").strip()) or "./frames"
    os.makedirs(out_dir, exist_ok=True)
    cfg["output_dir"] = out_dir
    cfg["video_fps"] = _prompt_int(
        "Video FPS (frames per second for the output video)", 1,
        lambda v: v >= 1, "FPS must be >= 1")

    print("\n" + "=" * 72)
    print("  PARAMETER SUMMARY")
    print("=" * 72)
    _LABEL_OVERRIDES = {
        # v41 REVERT: target_peak_dbm_array is the COMBINED TOTAL again.
        "target_peak_dbm_array": "target_peak_dbm_array (COMBINED TOTAL, "
                                  "max-hold peak across the whole array)",
    }
    for k, v in cfg.items():
        if k.startswith("_"):
            continue   # internal solver seeds, not user-facing settings
        label = _LABEL_OVERRIDES.get(k, k)
        print(f"  {label:45s}: {v}")
    print("=" * 72)

    return cfg


def upload_map_and_generate_mask(width_m=None, height_m=None, cell_size_m=0.5,
                                  cell_budget=260000):
    """
    v17: mask generation delegates entirely to SmartMapPreprocessor
    instead of the old flat darkness threshold.

    v21: width_m/height_m auto-derive at cell_size_m=0.5 m/px if not
    given, matching the reference site's fallback.

    v22 fix: width/height (and the solver grid) were being derived from
    the UPLOADED IMAGE'S RAW PIXEL COUNT -- a 1067x1474px upload times
    0.5 m/px silently became a 533x737m map (way too large for a campus
    block) AND a 1.57M-cell solve (slow). The reference never does this:
    it runs SmartMapPreprocessor-equivalent cleanup at full resolution
    (unchanged, not thrown away), THEN downsamples via compute_grid_size/
    cell_budget (default 260,000 cells, same as reference) BEFORE cell_
    size_m is applied -- so width/height and grid size both come from a
    sane, bounded resolution regardless of the upload's raw pixel count.
    This fixes the too-large-map (weak/dead-looking heatmap), the slow
    solve, AND gives a true anti-aliased occupancy field (not the earlier
    Gaussian-blur approximation) for effective-medium blending.
    """
    if not IN_COLAB:
        raise RuntimeError(
            "File upload requires Google Colab (google.colab.files). "
            "If running locally, instantiate SmartMapPreprocessor(path) "
            "yourself and call .process(width_m, height_m) directly.")

    print("\nUpload the FLOORPLAN image:")
    up_map = files.upload()
    if not up_map:
        raise ValueError("No floorplan uploaded.")
    map_name = list(up_map.keys())[0]

    tmp_path = os.path.join("/tmp", map_name)
    with open(tmp_path, "wb") as f:
        f.write(up_map[map_name])

    # Full-resolution cleanup UNCHANGED (park/green exclusion, two-stage
    # text removal, courtyard-preserving hole-fill) -- width/height passed
    # here are only used by SmartMapPreprocessor for its own metadata
    # printout, not for the solver grid, so a placeholder is fine.
    smp = SmartMapPreprocessor(tmp_path)
    smp.process(width_m=width_m or 100.0, height_m=height_m or 100.0)
    full_res_mask = smp.BinaryMask
    img = Image.fromarray(smp.OriginalImage)
    print(f"✓ Floorplan: {img.size[0]}×{img.size[1]} px, cleaned at full resolution")

    # NOW downsample the already-clean mask to a bounded solver grid --
    # same logic/math as the reference's compute_grid_size +
    # set_mask_from_canonical, applied to OUR mask instead of discarding
    # the SmartMapPreprocessor cleanup and starting over.
    solver_rows, solver_cols = compute_grid_size(full_res_mask.shape, cell_budget)
    occupancy, mask = downsample_mask_to_grid(full_res_mask, solver_rows, solver_cols)

    if width_m is None or height_m is None:
        width_m = solver_cols * cell_size_m
        height_m = solver_rows * cell_size_m
        print(f"   No map width/height given -- auto-derived from a "
              f"{cell_size_m} m/px cell size applied to the "
              f"{solver_cols}×{solver_rows} solver grid (cell_budget="
              f"{cell_budget}, same default the reference site uses): "
              f"{width_m:.1f} m × {height_m:.1f} m")

    building_pct = 100 * mask.mean()
    print(f"✓ Solver grid: {solver_cols}×{solver_rows} cells "
          f"({solver_rows*solver_cols:,} total, budget={cell_budget:,}) "
          f"-- downsampled from {img.size[0]}×{img.size[1]}px full-res mask")
    print(f"✓ Mask: {building_pct:.1f}% building, {100-building_pct:.1f}% air.")

    return img, mask, occupancy, width_m, height_m


print("✅  Input-collection helpers loaded. Call collect_user_inputs() and "
      "upload_map_and_generate_mask() in the next cell.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 7 — NODE PLACEMENT + FRAME GENERATION                            ║
# ║                                                                          ║
# ║   Uses NodePlacerOptimizer (per-cluster placement + prune/ILP           ║
# ║   optimization, percentile-based dead-zone threshold). mask.shape ==    ║
# ║   rssi_dbm_grid.shape always in this notebook (one shared grid), so     ║
# ║   result['finalNodes'] / result['transmitter'] come back directly in    ║
# ║   solver-grid coordinates — no img-space conversion needed anywhere.    ║
# ╚══════════════════════════════════════════════════════════════════════════╝


def run_node_placement(img, mask, rssi_dbm_grid, cfg):
    """
    Run NodePlacerOptimizer (new class) on the single-antenna baseline
    coverage. With ImageScaleFactor=1.0 and mask.shape == rssi_dbm_grid.shape
    (always true in this notebook -- one grid everywhere), the new class's
    internal cv2.resize calls are no-ops: result['img'] and every node
    position come back in EXACTLY the same (rows, cols) grid as
    rssi_dbm_grid and mask. No img-space<->solver-space conversion needed
    anywhere -- one coordinate system, confirmed by inspection of the class.

    img              : PIL.Image (uploaded floorplan)
    mask             : np.ndarray, SAME (rows, cols) as rssi_dbm_grid
                        (1=building, 0=air)
    rssi_dbm_grid    : np.ndarray from SingleAntennaSim.rssi_dbm_grid()
    cfg              : dict with the NodePlacerOptimizer parameters
                        collected in collect_user_inputs().
    """
    solver_rows, solver_cols = rssi_dbm_grid.shape

    # The FDM solvers (Cell 2/Cell 3) use a 20px "sponge" absorbing boundary
    # layer around all four edges of the grid, to stop outgoing waves from
    # reflecting back off the map's edges. That border is DELIBERATELY made
    # very lossy -- it's a numerical technique, not real-world signal -- so
    # it always reads as extremely weak dBm. Left as-is, that fake weak
    # ring gets treated as the "worst" part of the map: it skews the
    # percentile threshold to be far more pessimistic than the interior
    # data actually is, AND NodePlacerOptimizer happily places nodes there
    # (they're in "air"), which then never stop winning "weakest node"
    # searches since nothing beats a numerical artifact. Both issues are
    # fixed by excluding this border from consideration entirely:
    # Uses the single shared SPONGE_THICKNESS constant from Cell 1 — no more
    # separately hardcoded "20" that could drift out of sync with the solvers.
    interior = np.zeros_like(mask, dtype=bool)
    interior[SPONGE_THICKNESS:-SPONGE_THICKNESS, SPONGE_THICKNESS:-SPONGE_THICKNESS] = True

    p = cfg["_single_sim_params"]
    src_row, src_col = meters_to_pixel(p["antenna_x_m"], p["antenna_y_m"],
                                        p["h"], p["hy"])
    src_row = int(np.clip(src_row, 0, solver_rows - 1))
    src_col = int(np.clip(src_col, 0, solver_cols - 1))

    # Mark the border as "building" (not air) so NodePlacerOptimizer's own
    # existing air-only placement rule naturally excludes it -- no new
    # exclusion mechanism needed, just feed it accurate information.
    mask_for_placement = mask.copy()
    mask_for_placement[~interior] = 1

    npo = NodePlacerOptimizer()
    npo.ImageScaleFactor = 1.0   # keep everything on one shared grid
    # v19 fix: NodeCoverageRadius/MinDistPruning are PIXEL quantities inside
    # NodePlacerOptimizer, but pixels-per-meter varies with whatever
    # resolution the uploaded floorplan happens to be. Collecting these in
    # METERS (cfg) and converting via this map's own h here keeps node
    # spacing/coverage meaningful regardless of upload resolution -- a
    # high-res image no longer silently produces a tiny real-world radius
    # and dozens of crowded nodes.
    npo.NodeCoverageRadius = max(1, round(cfg["node_coverage_radius_m"] / p["h"]))
    npo.MinDistPruning = max(1, round(cfg["min_dist_pruning_m"] / p["h"]))
    print(f"   Node coverage radius: {cfg['node_coverage_radius_m']:.1f} m "
          f"-> {npo.NodeCoverageRadius}px | Min pruning distance: "
          f"{cfg['min_dist_pruning_m']:.1f} m -> {npo.MinDistPruning}px "
          f"(this map: {p['h']:.4f} m/px)")
    # Percentile computed over the INTERIOR only, so the sponge border's
    # fake extreme values can't skew what "the weakest 35%" even means.
    npo.DeadZoneThreshold_dBm = np.percentile(rssi_dbm_grid[interior].flatten(),
                                               cfg["dead_zone_percentile"])
    npo.MinClusterAreaPx = cfg["min_cluster_area"]
    npo.RadiiToTest = np.arange(10, 81, 10)
    npo.MaxCandidates = cfg["max_candidates"]
    npo.MaxNodes = cfg["max_nodes"]

    print(f"   Dead-zone threshold: {npo.DeadZoneThreshold_dBm:.1f} dBm "
          f"({cfg['dead_zone_percentile']}th percentile of this map's interior "
          f"RSSI, excluding the {SPONGE_THICKNESS}px absorbing boundary)")

    # v25 FIX: img was being passed at its FULL uploaded resolution (e.g.
    # 1067x1474 px from SmartMapPreprocessor), while mask/rssi_dbm_grid are
    # the DOWNSAMPLED solver grid (e.g. 445x584). NodePlacerOptimizer.run()
    # takes campusImg's own shape as (h_img, w_img) and does everything in
    # THAT resolution, then returns finalNodes in THOSE full-res pixel
    # coordinates. This function then clipped those large full-res
    # coordinates straight into [0, solver_cols-1]/[0, solver_rows-1] --
    # any node whose full-res x/y exceeded the (much smaller) solver grid
    # size got clamped straight to the boundary. That's why every node
    # landed on the rightmost column/bottom row regardless of where
    # NodePlacerOptimizer actually intended to place it. Resizing img to
    # the solver grid here makes img_array.shape == mask.shape ==
    # rssi_dbm_grid.shape, so ImageScaleFactor=1.0 is actually a true
    # single shared grid (as the docstring already claimed) and finalNodes
    # come back in real solver-grid coordinates -- no more edge-clamping.
    img_array = np.array(img)
    if img_array.shape[:2] != (solver_rows, solver_cols):
        img_array = cv2.resize(img_array, (solver_cols, solver_rows),
                                interpolation=cv2.INTER_AREA)
    result = npo.run(
        campusImg=img_array,
        stage1_out={"mag_db": rssi_dbm_grid, "src_x": src_col, "src_y": src_row},
        binaryMask=mask_for_placement
    )


    all_nodes = []
    for idx, (nx, ny) in enumerate(result["finalNodes"]):
        col = int(np.clip(round(nx), 0, solver_cols - 1))
        row = int(np.clip(round(ny), 0, solver_rows - 1))
        all_nodes.append({
            "id": idx,
            "_solver_rc": (row, col),
            "baseline_rssi": float(rssi_dbm_grid[row, col]),
        })

    # NodePlacerOptimizer decides WHERE to place nodes (percentile-based
    # dead-zone detection, then picks good coverage-representative points --
    # not necessarily the single worst pixel in a cluster). Separately, the
    # user decides which of those PLACED nodes are actually weak enough to
    # be worth steering the beam toward, via an explicit dBm cutoff.
    # v19 fix: if the user didn't set an explicit target threshold, derive
    # it from the weakest ACTUALLY-PLACED node (not the percentile value
    # itself -- that's computed over every interior pixel, a different
    # population than the specific pixels nodes landed on, so a blind
    # +2dB offset from the percentile can still miss and target zero
    # nodes). Using the real weakest placed node + margin guarantees at
    # least one node always qualifies by default.
    if cfg["target_rssi_threshold"] is None and all_nodes:
        weakest_placed = min(n["baseline_rssi"] for n in all_nodes)
        cfg["target_rssi_threshold"] = weakest_placed + 3.0
        print(f"   Target RSSI threshold: auto-set to "
              f"{cfg['target_rssi_threshold']:.1f} dBm "
              f"(weakest placed node {weakest_placed:.1f} dBm + 3 dB margin)")
    elif cfg["target_rssi_threshold"] is None:
        cfg["target_rssi_threshold"] = npo.DeadZoneThreshold_dBm + 2.0

    weak_nodes = [n for n in all_nodes if n["baseline_rssi"] < cfg["target_rssi_threshold"]]

    print(f"✓ NodePlacer: {len(result['nodesBeforeOpt'])} node(s) before "
          f"optimization -> {len(all_nodes)} after pruning "
          f"({result['coverageFraction']*100:.1f}% dead-zone coverage), "
          f"{len(weak_nodes)} below your {cfg['target_rssi_threshold']} dBm target threshold.")

    return {
        "result": result,
        "all_nodes": all_nodes,
        "weak_nodes": weak_nodes,
        "src_row": src_row,
        "src_col": src_col,
        "npo": npo,   # v41: exposed so the radius-sweep log can be plotted
    }


# ═══════════════════════════════════════════════════════════════════════════
#  FRAME GENERATION — all drawn directly in the solver's (rows, cols) grid,
#  displayed with extent=[0, real_width, real_height, 0] so axis labels show
#  meters. Node/antenna positions converted once via
#  node_img_px_to_solver_px / meters_to_pixel — no other conversions.
# ═══════════════════════════════════════════════════════════════════════════
FRAME_FIGSIZE = (12, 10)
FRAME_DPI = 100
FRAME_FACECOLOR = "#1e1e2e"
HEATMAP_ALPHA = 0.65   # matches reference engine's overlay alpha


def _draw_common_frame(mag_db, W, H, vmin=-100, vmax=-30, background_img=None):
    """
    v18: the equation is still solved entirely on the binary mask grid --
    nothing about the physics changes here. What changed is DISPLAY only:
    the heatmap is now alpha-overlaid on the actual original floorplan
    image (background_img, resized to the solver grid), matching the
    reference engine's render_heatmap_png() style, instead of being drawn
    alone on a flat dark canvas. If no background is supplied, falls back
    to the old flat canvas (keeps this function usable standalone).
    """
    fig, ax = plt.subplots(figsize=FRAME_FIGSIZE, dpi=FRAME_DPI,
                            facecolor=FRAME_FACECOLOR)
    ax.set_facecolor(FRAME_FACECOLOR)

    rows, cols = mag_db.shape
    if background_img is not None:
        bg_resized = cv2.resize(np.array(background_img), (cols, rows),
                                 interpolation=cv2.INTER_LINEAR)
        ax.imshow(bg_resized, extent=[0, W, H, 0], aspect="auto")
        alpha = HEATMAP_ALPHA
    else:
        alpha = 1.0

    im = ax.imshow(mag_db, cmap="jet", extent=[0, W, H, 0], vmin=vmin, vmax=vmax,
                    alpha=alpha)
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
    # v23: label matches what rssi_dbm_grid()/solve_for_angle() now actually
    # return (field-magnitude dB, reference-engine style) -- not literal RSSI dBm.
    cb.set_label("Signal Strength (dB)", color="#cdd6f4")
    plt.setp(cb.ax.yaxis.get_ticklabels(), color="#cdd6f4")
    ax.tick_params(colors="#cdd6f4")
    for sp in ax.spines.values():
        sp.set_edgecolor("#45475a")
    return fig, ax


def _rc_to_m(row, col, h, hy):
    return col * h, row * hy


def _text_box(ax, text):
    ax.text(0.02, 0.98, text, transform=ax.transAxes, va="top", ha="left",
            fontsize=10, family="monospace", color="#cdd6f4",
            bbox=dict(facecolor="#1e1e2e", alpha=0.85, pad=6,
                       boxstyle="round,pad=0.4"))


def generate_node_placement_frame(mask, all_nodes, weak_nodes,
                                   src_row, src_col, W, H, h, hy, out_path):
    """
    Frame 0 — shown BEFORE the baseline RSSI heatmap. Displays just the
    building layout (mask) with every placed node marked (red = above
    threshold, orange = below threshold / weak) and the antenna position,
    so the node-placement result is visible on its own before any coverage
    heatmap is drawn.
    """
    fig, ax = plt.subplots(figsize=FRAME_FIGSIZE, dpi=FRAME_DPI,
                            facecolor=FRAME_FACECOLOR)
    ax.set_facecolor(FRAME_FACECOLOR)

    # Building mask as a simple grayscale backdrop (1=building, 0=air).
    backdrop = np.where(mask == 1, 0.25, 0.05)
    ax.imshow(backdrop, cmap="gray", extent=[0, W, H, 0], vmin=0, vmax=1)

    weak_ids = {n["id"] for n in weak_nodes}
    for n in all_nodes:
        nr, nc = n["_solver_rc"]
        nx_m, ny_m = _rc_to_m(nr, nc, h, hy)
        color = "orange" if n["id"] in weak_ids else "red"
        ax.scatter(nx_m, ny_m, s=110, c=color, edgecolors="white",
                   linewidths=1.2, zorder=5)
        ax.text(nx_m + 0.015 * W, ny_m, f"#{n['id']:02d}", color="white",
                fontsize=8, zorder=6, va="center")

    ax_m, ay_m = _rc_to_m(src_row, src_col, h, hy)
    ax.scatter(ax_m, ay_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.tick_params(colors="#cdd6f4")
    for sp in ax.spines.values():
        sp.set_edgecolor("#45475a")

    ax.scatter([], [], s=110, c="red", edgecolors="white", label="Node (OK)")
    ax.scatter([], [], s=110, c="orange", edgecolors="white",
               label=f"Node (weak, will need coverage)")
    ax.scatter([], [], s=400, marker="*", c="magenta", edgecolors="white",
               label="Antenna")
    leg = ax.legend(loc="lower right", facecolor="#1e1e2e", labelcolor="#cdd6f4",
                     framealpha=0.85, fontsize=9)

    _text_box(ax, f"NODE PLACEMENT — {len(all_nodes)} node(s) placed\n"
                   f"{len(weak_nodes)} below RSSI threshold (need coverage)\n"
                   f"→ Baseline coverage heatmap next")

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


# v25 FIX: baseline used a FIXED dB color scale (cfg dbm_min/dbm_max) but
# the steering/final/max-hold frames each normalized to their OWN frame's
# min/max instead -- so identical dBm values rendered as different colors
# across frames, and the phased-array frames looked stronger/weaker than
# they actually were relative to the baseline just from rescaled color,
# not real signal difference. All frames now share ONE fixed scale, set
# once from cfg (same dbm_min/dbm_max the baseline already used), so a
# given color always means the same dBm everywhere.
FRAME_DBM_MIN = -90.0
FRAME_DBM_MAX = 20.0  # v28: reverted with the amp revert -- matches reference scale


def generate_baseline_frame(rssi_dbm_grid, all_nodes, weakest_node,
                             src_row, src_col, W, H, h, hy, out_path,
                             dbm_min=-100, dbm_max=-30, background_img=None):
    fig, ax = _draw_common_frame(rssi_dbm_grid, W, H, vmin=dbm_min, vmax=dbm_max,
                                  background_img=background_img)

    for n in all_nodes:
        nr, nc = n["_solver_rc"]
        nx_m, ny_m = _rc_to_m(nr, nc, h, hy)
        ax.scatter(nx_m, ny_m, s=80, c="red", edgecolors="white",
                   linewidths=1.0, zorder=5)

    ax_m, ay_m = _rc_to_m(src_row, src_col, h, hy)

    if weakest_node is not None:
        wr, wc = weakest_node["_solver_rc"]
        wx_m, wy_m = _rc_to_m(wr, wc, h, hy)
        ax.scatter(wx_m, wy_m, s=400, c="yellow", alpha=0.3, zorder=4)
        ax.scatter(wx_m, wy_m, s=200, c="yellow", edgecolors="white",
                   linewidths=1.5, zorder=6)
        ax.annotate("", xy=(wx_m, wy_m), xytext=(ax_m, ay_m),
                    arrowprops=dict(arrowstyle="->", color="cyan", lw=2), zorder=7)
        _text_box(ax, f"BASELINE — Single Antenna\n"
                       f"Target: #{weakest_node['id']:02d} | "
                       f"RSSI: {weakest_node['baseline_rssi']:.1f} dBm\n"
                       f"→ Replacing with phased array")
    else:
        _text_box(ax, "BASELINE — Single Antenna\nNo weak spots found.")

    ax.scatter(ax_m, ay_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def generate_final_labeled_frame(mag_db, theta_current, all_nodes,
                                  unreachable_ids, antenna_m, W, H, out_path,
                                  background_img=None):
    """
    Repeats the very last steering frame (same beam angle/heatmap, same
    beam wedge and antenna), but labels each node with the dB value read
    directly from THIS heatmap at that node's position -- what the phased
    array is actually delivering to each node at this final beam angle,
    not the static single-antenna baseline. Grey nodes (permanently
    excluded/unreachable) are shown grey, same as during steering. Shown
    once, right before the max-hold frame.
    """
    fig, ax = _draw_common_frame(mag_db, W, H, vmin=FRAME_DBM_MIN, vmax=FRAME_DBM_MAX,
                                  background_img=background_img)

    for n in all_nodes:
        nx_m, ny_m = n["_m"]
        nr, nc = n["_solver_rc"]
        val_at_this_angle = mag_db[nr, nc]
        color = "grey" if n["id"] in unreachable_ids else "red"
        ax.scatter(nx_m, ny_m, s=80, c=color, edgecolors="white",
                   linewidths=1.0, zorder=5)
        ax.annotate(f"{val_at_this_angle:.1f}", (nx_m, ny_m),
                    textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=7, color="white", zorder=9,
                    path_effects=[pe.withStroke(linewidth=2, foreground="black")])

    wedge = Wedge(antenna_m, 0.15 * min(W, H), theta_current - 25,
                  theta_current + 25, facecolor="#00ffff", alpha=0.12, zorder=3)
    ax.add_patch(wedge)

    ax.scatter(*antenna_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    beam_len = 0.1 * min(W, H)
    dx = beam_len * math.cos(math.radians(theta_current))
    dy = -beam_len * math.sin(math.radians(theta_current))
    ax.annotate("", xy=(antenna_m[0]+dx, antenna_m[1]+dy), xytext=antenna_m,
                arrowprops=dict(arrowstyle="->", color="white", lw=2), zorder=8)

    _text_box(ax, f"FINAL BEAM POSITION — {theta_current:03.0f}°\n"
                   f"Per-node dB labeled at this angle\n"
                   f"{len(all_nodes)} node(s) shown, {len(unreachable_ids)} grey/unreachable")

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def generate_max_hold_frame(max_hold_db, all_nodes, unreachable_ids,
                             antenna_m, W, H, out_path, background_img=None):
    """
    Final frame: full 360° max-hold coverage map -- the best (highest) dB
    value at each pixel across all 73 precomputed beam angles (matches the
    original PhasedArraySim.max_hold_coverage() exactly). Shown once, as the
    very last frame, after the steering loop completes. Grey = permanently
    excluded/unreachable node; red = everyone else still searchable.
    """
    fig, ax = _draw_common_frame(max_hold_db, W, H,
                                  vmin=FRAME_DBM_MIN, vmax=FRAME_DBM_MAX,
                                  background_img=background_img)

    for n in all_nodes:
        nx_m, ny_m = n["_m"]
        color = "grey" if n["id"] in unreachable_ids else "red"
        ax.scatter(nx_m, ny_m, s=80, c=color, edgecolors="white",
                   linewidths=1.0, zorder=5)

    ax.scatter(*antenna_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    _text_box(ax, f"PHASED ARRAY — TOTAL COMBINED (Max-Hold Peak)\n"
                   f"Best coherent-array dB per pixel across all beam angles\n"
                   f"(NOT per-element -- this is the full 5-element combined output)\n"
                   f"{len(all_nodes)} node(s) total, {len(unreachable_ids)} grey/unreachable")

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def generate_frame(mag_db, theta_current, theta_target, target, all_nodes,
                    unreachable_ids, iteration, n_iterations, antenna_m, W, H,
                    out_path, is_hold=False, background_img=None):
    """
    One steering-loop frame. Colors:
      - grey   : node was targeted directly before and still stayed below
                 the user's RSSI threshold -- permanently excluded, out of
                 our coverage, never searched again.
      - yellow : the current target.
      - red    : everyone else (still searchable, no permanent "served"
                 state -- can be retargeted any time it's the weakest).
    """
    fig, ax = _draw_common_frame(mag_db, W, H, vmin=FRAME_DBM_MIN, vmax=FRAME_DBM_MAX,
                                  background_img=background_img)

    for n in all_nodes:
        nx_m, ny_m = n["_m"]
        if n["id"] in unreachable_ids:
            ax.scatter(nx_m, ny_m, s=80, c="grey", edgecolors="white",
                       linewidths=1.0, zorder=5)
        elif n["id"] == target["id"]:
            ax.scatter(nx_m, ny_m, s=200, c="yellow", edgecolors="white",
                       linewidths=1.5, zorder=6)
        else:
            ax.scatter(nx_m, ny_m, s=80, c="red", edgecolors="white",
                       linewidths=1.0, zorder=5)

    wedge = Wedge(antenna_m, 0.15 * min(W, H), theta_current - 25,
                  theta_current + 25, facecolor="#00ffff", alpha=0.12, zorder=3)
    ax.add_patch(wedge)

    ax.scatter(*antenna_m, s=400, marker="*", c="magenta",
               edgecolors="white", linewidths=1.2, zorder=8)

    beam_len = 0.1 * min(W, H)
    dx = beam_len * math.cos(math.radians(theta_current))
    dy = -beam_len * math.sin(math.radians(theta_current))
    ax.annotate("", xy=(antenna_m[0]+dx, antenna_m[1]+dy), xytext=antenna_m,
                arrowprops=dict(arrowstyle="->", color="white", lw=2), zorder=8)

    hold_tag = " [HOLD]" if is_hold else ""
    _text_box(ax, f"Iteration: {iteration:02d} / {n_iterations}{hold_tag}\n"
                   f"Beam: {theta_current:03.0f}° → {theta_target:03.0f}°\n"
                   f"Target: #{target['id']:02d} | Baseline: {target['baseline_rssi']:.1f} dBm "
                   f"| Was: {target['current_rssi']:.1f} dBm")

    plt.tight_layout()
    fig.savefig(out_path, facecolor=FRAME_FACECOLOR)
    plt.close(fig)


def compile_video_from_frames(output_dir, fps, video_path=None):
    """
    Stitch every frame_XXXX.png in output_dir (in order) into an MP4 video
    at the given fps. The PNGs are kept as-is -- this just adds a video on
    top, it doesn't replace or delete the individual frame images.
    """
    if video_path is None:
        video_path = os.path.join(output_dir, "simulation_video.mp4")

    frame_files = sorted(
        f for f in os.listdir(output_dir)
        if f.startswith("frame_") and f.endswith(".png")
    )
    if not frame_files:
        print("   ⚠️  No frames found -- skipping video compilation.")
        return None

    first_frame = cv2.imread(os.path.join(output_dir, frame_files[0]))
    height, width = first_frame.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

    for fname in frame_files:
        frame = cv2.imread(os.path.join(output_dir, fname))
        if frame.shape[:2] != (height, width):
            frame = cv2.resize(frame, (width, height))
        writer.write(frame)
    writer.release()

    print(f"   ✓ Video written: {video_path} "
          f"({len(frame_files)} frames @ {fps} fps, "
          f"{len(frame_files)/fps:.1f}s runtime)")
    return video_path


print("✅  Node placement + frame generation functions loaded.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 10 — ANALYSIS (from analysis.zip, deduplicated + integrated)     ║
# ║                                                                          ║
# ║   Consolidated from 16 separate generator files (each a "class" +       ║
# ║   "main" pair) down to 3 real analyzer functions in one cell:           ║
# ║     - BeamSweepingAnalyzer   (2.3: pulls angles straight out of         ║
# ║       phased.angle_cache -- already computed by precompute_all_angles()║
# ║       in Cell 3 -- instead of recomputing/regenerating them)            ║
# ║     - PathLossSectorsAnalyzer (merges the original "Path Loss Sectors"  ║
# ║       AND "Single Path Loss Sectors" files -- those two were the same   ║
# ║       radial cross-section idea duplicated, one with phased+single      ║
# ║       overlaid and one single-only; single-only was 100% redundant)     ║
# ║     - PhasedArrayTradeoffAnalyzer (pure analytic N-vs-gain formula, no  ║
# ║       simulation data needed, kept close to original but anchored to    ║
# ║       this project's real per-element gain instead of a hardcoded       ║
# ║       constant)                                                         ║
# ║                                                                          ║
# ║   Removed entirely (per 2.1, overlap): every file's own SimParams /     ║
# ║   SectorData dataclass (this project already carries the same info in  ║
# ║   PhasedArraySim.params / SingleAntennaSim.params / angle_cache -- no   ║
# ║   need for a second parallel set of containers), every file's own       ║
# ║   get_master_fig() (was copy-pasted verbatim in 6+ files -- now one     ║
# ║   shared helper), and every file's "main" demo (those all built FAKE    ║
# ║   synthetic power maps with np.random noise -- useless here since this  ║
# ║   project has real solved data to plot instead).                        ║
# ║                                                                          ║
# ║   Deferred (out of scope for this pass, flagged not silently dropped):  ║
# ║   the Dashboard/Comprehensive/single-dashboard GUI wrapper files and    ║
# ║   the Pruning-Robustness 3D parameter-sweep surface -- those are        ║
# ║   presentation wrappers / expensive multi-run sweeps, not core          ║
# ║   algorithms, and would need a real scoping conversation before wiring  ║
# ║   in (the robustness sweep alone re-runs node placement dozens of       ║
# ║   times, which conflicts directly with the "don't recompute what's      ║
# ║   already stored" instruction unless deliberately scoped as a           ║
# ║   what-if study).                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

_analysis_fig = None


def _get_analysis_fig():
    """Single shared master figure -- replaces the identical get_master_fig()
    classmethod that was copy-pasted into 6+ of the original analysis files."""
    global _analysis_fig
    if _analysis_fig is None or not plt.fignum_exists(_analysis_fig.number):
        _analysis_fig = plt.figure("CUFE 5G Analysis", figsize=(14, 10))
        _analysis_fig.patch.set_facecolor("white")
    plt.figure(_analysis_fig.number)
    return _analysis_fig


def plot_beam_sweeping(phased_sim, background_img, target_angles=(15, 40, 60, 75)):
    """
    Shows 4 beam angles closest to target_angles. Pulls each heatmap
    directly from phased_sim.angle_cache -- already computed once by
    precompute_all_angles() in Cell 3 -- NOT recomputed here.
    """
    fig = _get_analysis_fig()
    fig.clear()
    p = phased_sim.params
    fig.suptitle(f"Beam Sweeping ({p['num_elements']} Elements)",
                 fontsize=14, fontweight="bold")

    rows, cols = p["rows"], p["cols"]
    img_resized = cv2.resize(np.array(background_img), (cols, rows)) \
        if background_img is not None else np.zeros((rows, cols, 3))

    for i, target in enumerate(target_angles):
        theta, mag_db = phased_sim.get_cached_angle(target)   # cache lookup, no solve

        ax = fig.add_subplot(2, 2, i + 1)
        ax.imshow(img_resized)
        im = ax.imshow(mag_db, alpha=0.65, cmap="jet", aspect="auto")
        real_peak = np.max(mag_db)
        im.set_clim(vmin=-80, vmax=max(15, real_peak))

        start_x, start_y = phased_sim.src_x[0], phased_sim.src_y[0]
        ax.plot(start_x, start_y, "rp", markersize=10, markerfacecolor="red")

        ax.set_title(f"Beam Directed at: {theta}°\nPeak: {real_peak:.1f} dBm",
                     fontsize=10, fontweight="bold")
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


def plot_pathloss_sectors(phased_sim, single_rssi_grid, single_sim_params,
                           target_angles=(15, 40, 60, 75)):
    """
    Radial cross-section (signal vs. distance) at 4 beam angles, single
    antenna vs. phased array overlaid. Merges the original "Path Loss
    Sectors" (combined) and "Single Path Loss Sectors" (single-only,
    100% redundant subset) files into one. Uses the already-solved
    rssi grids -- no recompute.
    """
    fig = _get_analysis_fig()
    fig.clear()
    fig.suptitle("Total Path Loss Across Sectors", fontsize=14, fontweight="bold")

    p = phased_sim.params
    rows, cols = p["rows"], p["cols"]
    h = p["h"]
    src_x, src_y = phased_sim.src_x[0], phased_sim.src_y[0]

    COLOR_SINGLE = (0.8500, 0.3250, 0.0980)
    COLOR_PHASED = (0.0, 0.4470, 0.7410)

    # v29 FIX: this used to sample phased_sim.get_cached_angle(target) --
    # the field for ONE fixed steered beam angle -- along a radial line at
    # that SAME angle, then labeled it "Phased Array (Total)". That is a
    # single-beam slice, not a total/combined result, and it is a
    # DIFFERENT metric than the heatmap's "TOTAL COMBINED (Max-Hold
    # Peak)" frame (best value per pixel across ALL 73 precomputed
    # angles). Comparing a max-hold heatmap (best-case envelope, always
    # >= any single angle by construction) against a single-angle radial
    # cross-section is comparing two different quantities under one
    # label -- that mismatch, not fake data, was why the heatmap looked
    # uniformly stronger while this chart showed the array losing in
    # places. Now both use the SAME max-hold matrix (phased_sim.result_matrix,
    # already computed by max_hold_coverage() earlier in main() before this
    # runs), so the chart is a true radial slice through the exact data the
    # heatmap renders -- genuinely comparable, no relabeling needed to make
    # it honest.
    if phased_sim.result_matrix is None:
        phased_sim.max_hold_coverage()
    mag_db_phased_total = phased_sim.result_matrix

    for i, target in enumerate(target_angles):
        theta_rad = np.radians(target)

        max_r = int(np.ceil(np.sqrt(rows**2 + cols**2)))
        r_px = np.arange(max_r + 1)
        x = np.round(src_x + r_px * np.cos(theta_rad)).astype(int)
        y = np.round(src_y + r_px * np.sin(theta_rad)).astype(int)
        valid = (x >= 0) & (x < cols) & (y >= 0) & (y < rows)
        x, y, r_valid = x[valid], y[valid], r_px[valid]
        r_m = r_valid * h

        val_phased = mag_db_phased_total[y, x]
        val_single = single_rssi_grid[y, x]

        ax = fig.add_subplot(2, 2, i + 1)
        ax.plot(r_m, val_single, linewidth=1.5, color=COLOR_SINGLE, label="Single Antenna")
        ax.plot(r_m, val_phased, linewidth=1.5, color=COLOR_PHASED, label="Phased Array (Max-Hold Total)")
        ax.set_xlabel("Distance from source (m)", fontweight="bold")
        ax.set_ylabel("Signal Strength (dBm)", fontweight="bold")
        ax.set_title(f"Cross-section at {target}°")
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)
        _lo = min(val_single.min(), val_phased.min()) - 5
        _hi = max(val_single.max(), val_phased.max()) + 5
        ax.set_ylim([_lo, _hi])   # v41: fully data-adaptive, no hardcoded floor

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


def plot_phased_array_tradeoff(single_sim_params, num_elements=5):
    """
    Pure analytic plot -- total peak array gain vs. number of elements
    (gain = single-element gain + 20*log10(N)). No simulation data needed.
    Anchored to this project's REAL single-element gain
    (single_sim_params['tx_gain_dbi']) instead of the original file's
    hardcoded 5.02 dB placeholder constant.
    """
    fig = _get_analysis_fig()
    fig.clear()
    fig.suptitle("Phased Array Gain", fontsize=14, fontweight="bold")

    element_gain_db = single_sim_params["tx_gain_dbi"]
    n_range = np.arange(1, 16)
    total_peak_db = element_gain_db + 20 * np.log10(n_range)

    ax = fig.add_subplot(111)
    ax.plot(n_range, total_peak_db, "-ob", linewidth=2.5, markersize=7,
            markerfacecolor="blue", label="Total Peak Gain (Logarithmic Growth)")

    target_gain = total_peak_db[num_elements - 1]
    ax.plot(num_elements, target_gain, "pg", markersize=14, markerfacecolor="green",
            label="Operating Point")
    ax.annotate(f"Operating Point\n({num_elements} Elements, {target_gain:.1f} dB)",
                (num_elements + 0.3, target_gain - 0.5),
                fontweight="bold", fontsize=10)

    ax.set_title("Total Peak Gain vs. Number of Elements", fontweight="bold", fontsize=13)
    ax.set_xlabel("Number of Antenna Elements (N)", fontweight="bold")
    ax.set_ylabel("Total Peak Gain (dB)", fontweight="bold")
    ax.set_xticks(n_range)
    ax.set_ylim([0, total_peak_db.max() + 5])
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


def plot_signal_quality_distribution(single_grid, phased_grid, mask=None):
    """
    Bar chart: % of coverage AREA falling into each signal-quality bucket
    (Dead / Poor / Good / Excellent), single antenna vs. total phased
    array. Uses the SAME already-solved, already-calibrated dB grids
    everything else in the pipeline uses (single_grid = rssi_grid,
    phased_grid = phased_sim.result_matrix, both post-Cell-8 calibration)
    -- no synthetic/random data anywhere, only real solved values.

    Percentages are computed over the free-space (non-building) pixels
    only, same population NodePlacerOptimizer's own percentile logic
    uses, since building-interior pixels aren't real coverage area.
    """
    fig = _get_analysis_fig()
    fig.clear()

    if mask is not None:
        free = (mask == 0)
        s_vals = single_grid[free]
        p_vals = phased_grid[free]
    else:
        s_vals = single_grid.ravel()
        p_vals = phased_grid.ravel()

    bins = [(-np.inf, -80, "Dead\n(< -80)"),
            (-80, -60, "Poor\n(-80 to -60)"),
            (-60, -40, "Good\n(-60 to -40)"),
            (-40, np.inf, "Excellent\n(> -40)")]

    single_pct, phased_pct, labels = [], [], []
    for lo, hi, label in bins:
        s_frac = np.mean((s_vals >= lo) & (s_vals < hi)) * 100.0
        p_frac = np.mean((p_vals >= lo) & (p_vals < hi)) * 100.0
        single_pct.append(s_frac)
        phased_pct.append(p_frac)
        labels.append(label)

    COLOR_SINGLE = (0.8500, 0.3250, 0.0980)
    COLOR_PHASED = (0.0, 0.4470, 0.7410)

    x = np.arange(len(labels))
    width = 0.35
    ax = fig.add_subplot(111)
    ax.bar(x - width / 2, single_pct, width, label="Single Antenna",
           color=COLOR_SINGLE, edgecolor="black", linewidth=0.5)
    ax.bar(x + width / 2, phased_pct, width, label="Total Phased Array",
           color=COLOR_PHASED, edgecolor="black", linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Area Percentage (%)", fontweight="bold")
    ax.set_title("Signal Quality Distribution", fontweight="bold", fontsize=13)
    ax.legend(loc="upper left")
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_ylim([0, max(max(single_pct), max(phased_pct)) + 10])  # data-adaptive

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


def plot_3d_tradeoff_landscape(npo, output_dir):
    """
    Interactive, rotatable 3D trade-off landscape: Node Radius (hardware
    spec) vs Required Nodes (system cost) vs resulting Coverage Area (%).

    v41: the connecting path is now spline-smoothed (line only -- no
    filled curtain panels), which is safe here in a way the original
    matplotlib curtain wasn't: a plain 3D *line* has nothing to
    depth-sort, so there's no self-crossing/tangled-panel artifact,
    unlike Poly3DCollection quads. The smoothing parameter is the sample
    INDEX (t = 0, 1, 2, ...), not radius/nodes/coverage themselves, so it
    only interpolates a smooth path BETWEEN the real measured points --
    it can't move, skip, or invent a measured value. Red/jet-colored
    markers remain exactly the literal (radius, estNodes, coverage%)
    triples NodePlacerOptimizer's own radius sweep logged (v37, no
    math/logic touched); only the connecting curve and its vertical
    stems are smoothed geometry.
    """
    log = sorted(getattr(npo, "radius_sweep_log", []), key=lambda t: t[0])
    if len(log) < 2:
        print("   (skipped 3D trade-off landscape: fewer than 2 radius-sweep "
              "samples were logged)")
        return None

    try:
        import plotly.graph_objects as go
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install",
                         "--break-system-packages", "-q", "plotly"], check=True)
        import plotly.graph_objects as go

    from scipy.interpolate import make_interp_spline

    radii = np.array([r for r, _, _ in log])
    nodes = np.array([n for _, n, _ in log])
    cov = np.array([c for _, _, c in log])
    span = cov.max() - cov.min()
    floor = cov.min() - (span * 0.15 if span > 0 else 1.0)

    # Spline through the sample INDEX (t), not through radius/nodes/
    # coverage directly -- guarantees the smooth curve passes exactly
    # through every real measured point, in the order they were swept,
    # and never extrapolates beyond them.
    t = np.arange(len(log))
    t_fine = np.linspace(0, len(log) - 1, 400)
    k = min(3, len(log) - 1)
    radii_s = make_interp_spline(t, radii, k=k)(t_fine)
    nodes_s = make_interp_spline(t, nodes, k=k)(t_fine)
    cov_s = make_interp_spline(t, cov, k=k)(t_fine)

    fig = go.Figure()

    # v41: vertical stems now carry the same red(high)->blue(low) Jet
    # gradient as the reference image, colored by actual z (coverage)
    # height along each stem -- not by relative position, so the color
    # at a given height is consistent across all stems, matching the
    # reference. Each stem is its own independent line, so there's still
    # nothing to depth-sort across stems (no filled-panel artifact risk).
    for r, n, c in zip(radii, nodes, cov):
        z_stem = np.linspace(floor, c, 30)
        fig.add_trace(go.Scatter3d(
            x=[r] * len(z_stem), y=[n] * len(z_stem), z=z_stem, mode="lines",
            line=dict(color=z_stem, colorscale="Jet",
                       cmin=cov.min(), cmax=cov.max(), width=6),
            showlegend=False, hoverinfo="skip"))

    # Smooth connecting curve (line only, no filled panels -> no
    # depth-sort artifacts), colored along its length by coverage.
    fig.add_trace(go.Scatter3d(
        x=radii_s, y=nodes_s, z=cov_s, mode="lines",
        line=dict(color=cov_s, colorscale="Jet", width=8),
        showlegend=False, hoverinfo="skip"))

    # Real measured points on top, as markers only.
    fig.add_trace(go.Scatter3d(
        x=radii, y=nodes, z=cov, mode="markers",
        marker=dict(size=7, color=cov, colorscale="Jet",
                     colorbar=dict(title="Coverage (%)"),
                     line=dict(color="black", width=1)),
        name="Measured sweep points",
        hovertemplate="Radius=%{x}<br>Nodes=%{y}<br>Coverage=%{z:.1f}%<extra></extra>"))

    fig.update_layout(
        title="Unified 3D Trade-off Landscape<br><sub>Real NodePlacerOptimizer "
              "radius sweep -- drag to rotate</sub>",
        scene=dict(
            xaxis_title="Node Radius (Hardware Specs)",
            yaxis_title="Required Nodes (System Cost)",
            zaxis_title="Coverage Area (%)",
        ),
        width=1000, height=800,
    )

    html_path = os.path.join(output_dir, "09_analysis_3d_tradeoff_landscape.html")
    fig.write_html(html_path, include_plotlyjs="cdn")
    print(f"   ✓ 3D trade-off landscape (interactive, rotatable) saved to "
          f"{html_path}")

    png_path = os.path.join(output_dir, "09_analysis_3d_tradeoff_landscape.png")
    try:
        fig.write_image(png_path, width=1200, height=900, scale=2)
        print(f"   ✓ Static preview (Plotly/kaleido) saved to {png_path}")
    except Exception as e:
        print(f"   (Plotly PNG export unavailable -- {e}; "
              f"falling back to a matplotlib static preview instead)")
        from mpl_toolkits.mplot3d.art3d import Line3DCollection
        import matplotlib.cm as cm
        import matplotlib.colors as mcolors

        fallback_fig = _get_analysis_fig()
        fallback_fig.clear()
        ax = fallback_fig.add_subplot(111, projection="3d")

        # Single normalization (real data's min/max) shared by the stems,
        # the smoothed curve, and the markers, so color <-> coverage-value
        # mapping is consistent across the whole figure.
        norm = mcolors.Normalize(vmin=cov.min(), vmax=cov.max())

        # v41: gradient stems (colored by actual z/coverage height)
        # instead of plain gray, matching the reference image's look.
        for r, n, c in zip(radii, nodes, cov):
            z_stem = np.linspace(floor, c, 30)
            stem_pts = np.array([np.full_like(z_stem, r), np.full_like(z_stem, n), z_stem]).T
            stem_pts = stem_pts.reshape(-1, 1, 3)
            stem_segs = np.concatenate([stem_pts[:-1], stem_pts[1:]], axis=1)
            stem_lc = Line3DCollection(stem_segs, cmap="jet", norm=norm, linewidths=2.5)
            stem_lc.set_array((z_stem[:-1] + z_stem[1:]) / 2)
            ax.add_collection3d(stem_lc)

        # Smoothed line as colored segments -- a plain 3D polyline, so
        # (unlike the old curtain) there's nothing to depth-sort wrong.
        pts = np.array([radii_s, nodes_s, cov_s]).T.reshape(-1, 1, 3)
        segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
        lc = Line3DCollection(segs, cmap="jet", norm=norm, linewidths=3)
        lc.set_array((cov_s[:-1] + cov_s[1:]) / 2)
        ax.add_collection3d(lc)

        sc = ax.scatter(radii, nodes, cov, c=cov, cmap="jet", s=45,
                         edgecolors="black", linewidths=0.8, depthshade=False)
        ax.set_xlabel("Node Radius (Hardware Specs)", fontweight="bold")
        ax.set_ylabel("Required Nodes (System Cost)", fontweight="bold")
        ax.set_zlabel("Coverage Area (%)", fontweight="bold")
        ax.set_title("Unified 3D Trade-off Landscape\n(static fallback preview, smoothed)",
                     fontweight="bold")
        ax.set_xlim(radii.min(), radii.max())
        ax.set_ylim(nodes.min(), nodes.max())
        ax.set_zlim(floor, cov.max() * 1.05 if cov.max() > 0 else 1.0)
        fallback_fig.colorbar(sc, ax=ax, shrink=0.6, label="Coverage (%)")
        fallback_fig.savefig(png_path, facecolor="white")
        print(f"   ✓ Static preview (matplotlib fallback) saved to {png_path}")

    return fig

def run_analysis_suite(phased_sim, single_sim, single_rssi_grid, mask, img, npo, output_dir):
    """
    Runs all 3 analyzers on already-computed data (angle_cache, rssi grid,
    params) -- nothing here triggers a new solve. Saves each as its own
    PNG alongside the main pipeline's frames.
    """
    print("\n" + "=" * 72)
    print("ANALYSIS SUITE (real cached data, no recomputation)")
    print("=" * 72)

    fig1 = plot_beam_sweeping(phased_sim, img)
    fig1.savefig(os.path.join(output_dir, "05_analysis_beam_sweeping.png"),
                 facecolor="white")
    print("   ✓ Beam sweeping (4 angles from angle_cache)")

    fig2 = plot_pathloss_sectors(phased_sim, single_rssi_grid, single_sim.params)
    fig2.savefig(os.path.join(output_dir, "06_analysis_pathloss_sectors.png"),
                 facecolor="white")
    print("   ✓ Path loss sectors (single vs. phased, same 4 angles)")

    fig3 = plot_phased_array_tradeoff(single_sim.params, num_elements=5)
    fig3.savefig(os.path.join(output_dir, "07_analysis_array_tradeoff.png"),
                 facecolor="white")
    print("   ✓ Phased array N-vs-gain tradeoff (analytic)")

    # v41: new analysis -- signal quality distribution, real data from the
    # already-solved (and calibrated) single-antenna grid + phased-array
    # max-hold total, no fake/random data.
    if phased_sim.result_matrix is None:
        phased_sim.max_hold_coverage()
    fig4 = plot_signal_quality_distribution(
        single_rssi_grid, phased_sim.result_matrix, mask)
    fig4.savefig(os.path.join(output_dir, "08_analysis_signal_quality_distribution.png"),
                 facecolor="white")
    print("   ✓ Signal quality distribution (real solved-grid area percentages)")

    # v41: plot_3d_tradeoff_landscape now saves its own files (interactive
    # .html + best-effort static .png) internally, since it's a Plotly
    # figure, not a matplotlib one -- doesn't use fig.savefig().
    plot_3d_tradeoff_landscape(npo, output_dir)

    plt.close("all")
    print("=" * 72)


print("✅  Analysis suite loaded (BeamSweeping, PathLossSectors, "
      "PhasedArrayTradeoff -- deduplicated, real data, no recompute).")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   CELL 8 — MAIN LOOP                                                    ║
# ║                                                                          ║
# ║   Pipeline (reordered per request):                                    ║
# ║     1. Collect params, upload floorplan, auto-generate mask internally ║
# ║     2. Build the Helmholtz system matrix ONCE (shared by both solvers) ║
# ║     3. Single-antenna solve -> baseline RSSI heatmap (own frame)       ║
# ║     4. NodePlacerOptimizer -> weak-spot nodes (own frame)              ║
# ║     5. Phased array installed at SAME position, reusing the SAME       ║
# ║        factorized matrix from step 2 (no second factorization)         ║
# ║     6. Max-hold coverage frame (own frame) — shown BEFORE the          ║
# ║        real-time video, not buried as the video's last frame           ║
# ║     7. Real-time beam-steering LOOP -- these frames (and only these)   ║
# ║        are compiled into the output video                             ║
# ║                                                                          ║
# ║   Math/physics: UNCHANGED throughout.                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝


def _rescale_to_target_peak(grid, target_peak_db):
    """Shift an already-solved dB grid by one uniform additive offset so
    its peak lands at target_peak_db.

    Physically exact, not a fake/synthetic edit: mag_db = 20*log10(|E|),
    and |E| scales LINEARLY with the source amplitude used in the solve
    (it's a linear PDE, A*E = -b). So for any amplitude ratio r:
        20*log10(r*|E|) = 20*log10(r) + 20*log10(|E|) = 20*log10(r) + mag_db
    i.e. adding a constant to mag_db everywhere is IDENTICAL, pixel for
    pixel, to having solved the same PDE with a different (scaled) source
    amplitude from the start. Nothing about the field's shape, decay, or
    relative values changes -- only the reference level, exactly the way
    turning a real transmitter's power up or down would.
    Returns (rescaled_grid, offset_db_applied, raw_measured_peak_db).
    """
    measured_peak = float(np.max(grid))
    offset_db = target_peak_db - measured_peak
    return grid + offset_db, offset_db, measured_peak


def main():
    cfg = collect_user_inputs_part1()
    img, mask, occupancy, width_m, height_m = upload_map_and_generate_mask()   # auto: 0.5 m/px on a bounded solver grid
    cfg["width"], cfg["height"] = width_m, height_m
    cfg = collect_user_inputs_part2(cfg)

    W, H = cfg["width"], cfg["height"]
    rows, cols = mask.shape
    h  = W / cols
    hy = H / rows

    lam = 3e8 / cfg["freq_hz"]
    ppw_actual = lam / min(h, hy)
    print(f"\nMask resolution: {cols}×{rows} px over {W}×{H} m "
          f"-> grid spacing {min(h,hy):.3f} m -> {ppw_actual:.1f} points/wavelength "
          f"at {cfg['freq_hz']/1e6:.1f} MHz.")
    if ppw_actual < cfg["ppw_target"]:
        print(f"  ⚠️  This is below your target PPW ({cfg['ppw_target']}). "
              f"Consider a higher-resolution floorplan image or a lower "
              f"frequency for a sharper simulation.")

    # ── Shared Helmholtz system matrix — built ONCE, used by BOTH the ──────
    # single-antenna solver and the phased array (see Cell 1). Previously
    # each solver factorized its own copy of an identical matrix; this was
    # the single most expensive redundant step in the notebook.
    # v22: built from `occupancy` (true anti-aliased fractional occupancy
    # on the bounded solver grid), not the hard mask -- `mask` (hard,
    # same resolution) is still what SingleAntennaSim/PhasedArraySim/
    # NodePlacerOptimizer use for source-snapping, placement, and frames.
    print("\n" + "=" * 72)
    print("STEP 1: Building shared Helmholtz system matrix (once)")
    print("=" * 72)
    shared_solve, shared_grid_params = build_helmholtz_solver(
        occupancy, W, cfg["freq_hz"], cfg["alpha_air"], cfg["alpha_eff"])

    # ── 1. Single-antenna baseline ──────────────────────────────────────
    print("\n" + "=" * 72)
    print("STEP 2: Single-antenna baseline solve")
    print("=" * 72)
    single_sim = SingleAntennaSim(
        binary_mask=mask, real_width=W, real_height=H,
        tx_power_dbm=cfg["_single_antenna_probe_amp"], tx_gain_dbi=2.15, rx_gain_dbi=0.0,
        antenna_x=cfg["antenna_x"], antenna_y=cfg["antenna_y"],
        f_sim=cfg["freq_hz"], alpha_air=cfg["alpha_air"], alpha_eff=cfg["alpha_eff"],
        shared_solve=shared_solve, shared_grid_params=shared_grid_params)
    single_sim.place_source()
    rssi_grid = single_sim.rssi_dbm_grid()
    cfg["_single_sim_params"] = single_sim.params

    # v41: calibrate to the user's requested peak dB (see _rescale_to_
    # target_peak docstring -- exact linear rescale, same physics).
    rssi_grid, _off_s, _raw_peak_s = _rescale_to_target_peak(
        rssi_grid, cfg["target_peak_dbm_single"])
    print(f"  ✓ Single-antenna calibrated to target peak "
          f"{cfg['target_peak_dbm_single']:.1f} dB "
          f"(raw solve peak was {_raw_peak_s:.1f} dB, applied {_off_s:+.1f} dB "
          f"offset uniformly).")

    src_row0, src_col0 = single_sim.src_row, single_sim.src_col
    antenna_m = _rc_to_m(src_row0, src_col0, h, hy)

    # ── 1b. Single-antenna baseline heatmap frame (its own standalone image,
    # shown before node placement) ───────────────────────────────────────
    print("\n" + "=" * 72)
    print("STEP 2b: Single-antenna baseline heatmap (standalone frame)")
    print("=" * 72)
    generate_baseline_frame(
        rssi_grid, [], None, src_row0, src_col0, W, H, h, hy,
        os.path.join(cfg["output_dir"], "01_single_antenna_heatmap.png"),
        dbm_min=cfg["dbm_min"], dbm_max=cfg["dbm_max"], background_img=img)

    # ── 2. Node placement on baseline coverage ──────────────────────────
    print("\n" + "=" * 72)
    print("STEP 3: Node placement (NodePlacerOptimizer)")
    print("=" * 72)
    placement = run_node_placement(img, mask, rssi_grid, cfg)
    all_nodes = placement["all_nodes"]
    weak_nodes = placement["weak_nodes"]
    src_row, src_col = placement["src_row"], placement["src_col"]

    for n in all_nodes:
        rc = n["_solver_rc"]
        n["_m"] = _rc_to_m(rc[0], rc[1], h, hy)
        n["current_rssi"] = n["baseline_rssi"]

    print("\n" + "=" * 72)
    print("STEP 3b: Node-placement frame (standalone)")
    print("=" * 72)
    generate_node_placement_frame(
        mask, all_nodes, weak_nodes, src_row, src_col, W, H, h, hy,
        os.path.join(cfg["output_dir"], "02_node_placement.png"))

    if not weak_nodes:
        if not all_nodes:
            print(f"\n  ✅ No dead-zone nodes found at the "
                  f"{cfg['dead_zone_percentile']}th percentile threshold — "
                  f"coverage already looks adequate. Nothing to steer toward.")
        else:
            print(f"\n  ✅ NodePlacer placed {len(all_nodes)} node(s), but "
                  f"none are below your target threshold of "
                  f"{cfg['target_rssi_threshold']} dBm (weakest placed node: "
                  f"{min(n['baseline_rssi'] for n in all_nodes):.1f} dBm). "
                  f"Nothing to steer toward.")
        print(f"\n  ✅ DONE — standalone frames written to {cfg['output_dir']} "
              f"(no phased-array/video step, nothing to steer toward).")
        return

    # ── 3. Phased array setup — SAME position, SAME grid, REUSES the ───────
    # shared factorized matrix built in STEP 1 (no second factorization).
    print("\n" + "=" * 72)
    print("STEP 4: Phased array setup (reusing shared system matrix, "
          "same raw-amplitude scale as the single antenna -- v24, no "
          "calibration step exists anymore on either solver)")
    print("=" * 72)
    # v24 FIX: single_sim.params["_calibration_ratio"] no longer exists --
    # SingleAntennaSim._calibrate_source_amp() was removed (reference-exact
    # raw amplitude, no free-space calibration). calib_ratio is a dead
    # concept now; PhasedArraySim's calib_ratio arg is left at its 1.0
    # default (no-op) instead of being looked up.
    phased = PhasedArraySim(
        raw_image=img, binary_mask=mask, real_width=W, real_height=H,
        tx_power_dbm=cfg["_array_element_probe_amp"],
        tx_gain_dbi=2.15, rx_gain_dbi=0.0,   # same defaults as SingleAntennaSim
        antenna_x=cfg["antenna_x"], antenna_y=cfg["antenna_y"],
        f_sim=cfg["freq_hz"], alpha_air=cfg["alpha_air"],
        alpha_eff=cfg["alpha_eff"], step_angle=5,
        shared_solve=shared_solve, shared_grid_params=shared_grid_params)
    phased.process_map()
    phased.place_sources()
    phased.precompute_all_angles()   # reuses shared_solve, no rebuild

    # v41 REVERT: back to v33 behavior on explicit request -- cfg[
    # "target_peak_dbm_array"] IS the combined max-hold TOTAL directly
    # again (the v41 "per-element + 20*log10(N) array gain" interpretation
    # is removed). Every cached angle (and the max-hold total) came from
    # the SAME fixed per-element amplitude, so one offset is exact for all
    # of them at once (see _rescale_to_target_peak docstring) -- shifting
    # here, before max-hold/frames/analysis are computed, keeps every
    # downstream consumer (heatmaps, beam-sweeping, path-loss sectors,
    # real-time loop) automatically consistent, nothing re-solved, no
    # relative shape changed.
    _raw_peak_a = float(max(np.max(g) for g in phased.angle_cache.values()))
    _off_a = cfg["target_peak_dbm_array"] - _raw_peak_a
    for _theta in list(phased.angle_cache.keys()):
        phased.angle_cache[_theta] = phased.angle_cache[_theta] + _off_a
    print(f"  ✓ Phased array calibrated to target COMBINED TOTAL peak "
          f"{cfg['target_peak_dbm_array']:.1f} dB "
          f"(raw solve peak was {_raw_peak_a:.1f} dB, applied {_off_a:+.1f} dB "
          f"offset uniformly to all {len(phased.angle_cache)} cached angles).")

    # ── 4. Max-hold coverage frame — shown NOW, before the real-time video,
    # not tacked on as the video's final frame. ─────────────────────────
    print("\n" + "=" * 72)
    print("STEP 5: Max-hold coverage frame (standalone, shown before video)")
    print("=" * 72)
    max_hold_db = phased.max_hold_coverage()   # recomputed from the now-calibrated angle_cache
    generate_max_hold_frame(
        max_hold_db, all_nodes, set(), antenna_m, W, H,
        os.path.join(cfg["output_dir"], "03_max_hold_coverage.png"), background_img=img)

    # ── 5b. Analysis suite -- BeamSweeping/PathLossSectors/Tradeoff, all
    # reading from phased.angle_cache / rssi_grid already computed above,
    # nothing here re-solves anything. ──────────────────────────────────
    run_analysis_suite(phased, single_sim, rssi_grid, mask, img,
                        placement["npo"], cfg["output_dir"])

    # ── 5. Real-time steering loop (instant electronic steering) — ONLY
    # these frames go into the video. ───────────────────────────────────
    print("\n" + "=" * 72)
    print("STEP 6: Real-time beam steering (video frames)")
    print("=" * 72)
    N = cfg["n_iterations"]
    theta_current = 0.0
    unreachable_ids = set()
    frame_num = 0

    for iteration in range(1, N + 1):
        searchable = [n for n in weak_nodes if n["id"] not in unreachable_ids]
        if not searchable:
            print(f"  All weak nodes are unreachable (grey) -- stopping early "
                  f"after {iteration - 1} iteration(s).")
            break

        target = min(searchable, key=lambda n: n["current_rssi"])
        tx_m, ty_m = target["_m"]

        dx = tx_m - antenna_m[0]
        dy = -(ty_m - antenna_m[1])
        theta_target_raw = math.degrees(math.atan2(dy, dx)) % 360
        theta_target, mag_db = phased.get_cached_angle(theta_target_raw)
        theta_current = theta_target

        generate_frame(
            mag_db, theta_current, theta_target, target, all_nodes,
            unreachable_ids, iteration, N, antenna_m, W, H,
            os.path.join(cfg["output_dir"], f"frame_{frame_num:04d}.png"),
            is_hold=True, background_img=img)
        frame_num += 1

        nr, nc = target["_solver_rc"]
        delivered_rssi = float(mag_db[nr, nc])
        target["current_rssi"] = delivered_rssi
        if delivered_rssi < cfg["target_rssi_threshold"]:
            unreachable_ids.add(target["id"])
            print(f"  Iteration {iteration}/{N}: targeted node #{target['id']:02d} "
                  f"-> still {delivered_rssi:.1f} dBm (below "
                  f"{cfg['target_rssi_threshold']} dBm threshold) -- "
                  f"marked GREY, excluded from now on.")
        else:
            print(f"  Iteration {iteration}/{N}: targeted node #{target['id']:02d} "
                  f"-> {delivered_rssi:.1f} dBm, now above threshold.")

    # ── 6. Final labeled frame (last beam position, standalone) ────────────
    print("\n" + "=" * 72)
    print("STEP 7: Final labeled frame (standalone)")
    print("=" * 72)
    _, final_mag_db = phased.get_cached_angle(theta_current)
    generate_final_labeled_frame(
        final_mag_db, theta_current, all_nodes, unreachable_ids, antenna_m, W, H,
        os.path.join(cfg["output_dir"], "04_final_labeled.png"), background_img=img)

    # ── 7. Compile ONLY the steering-loop frames into the real-time video ──
    print("\n" + "=" * 72)
    print("STEP 8: Compiling real-time steering video")
    print("=" * 72)
    video_path = compile_video_from_frames(cfg["output_dir"], cfg["video_fps"])

    print("\n" + "=" * 72)
    print(f"  ✅ DONE — output written to {cfg['output_dir']}:")
    print(f"     01_single_antenna_heatmap.png")
    print(f"     02_node_placement.png")
    print(f"     03_max_hold_coverage.png")
    print(f"     04_final_labeled.png")
    print(f"     frame_0000.png ... frame_{frame_num-1:04d}.png  (steering-loop frames)")
    if video_path:
        print(f"     video: {video_path}  (steering loop ONLY, {frame_num} frames)")
    print("=" * 72)


if __name__ == "__main__":
    main()
